# CLEETS-SMART Prediction Notebook

This notebook starts from the temporal CLEETS knowledge graph and keeps the workflow deliberately simple.

It does four things:

1. Uses SPARQL to extract integer EV keepership and EV charger counts.
2. Keeps only the 22 Welsh LADs and yearly observations from 2019 to 2025.
3. Trains forecasting, machine-learning and deep-learning models to predict both targets:
   - EV keepership
   - EV charger counts
4. Forecasts both targets year-by-year to 2045 and plots time series.

No keeper to charger ratio is used during modelling. Ratios were calculated later from the two predicted count columns.


In [ ]:
!pip install -q rdflib pandas numpy scikit-learn matplotlib plotly torch openpyxl gdown

# Standard library
import re
import math
import copy
import unicodedata
from urllib.parse import quote

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import plotly.express as px

# RDF / Semantic Web
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

# Machine learning
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F

# Google Colab utilities
import gdown
from google.colab import files
import google.colab.data_table

# ============================================================
# 1. Notebook configuration
# ============================================================

google.colab.data_table.enable_dataframe_formatter()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Libraries loaded successfully.")

# Download and access the data

This cell downloads and also loads two electric vehicle (EV) datasets from Google Drive (temporary storage by Naeima) into pandas DataFrames for analysis.

How the code works: First, it defines the URLs of the EV keepership and EV charger datasets stored in Google Drive. Since Google Drive sharing links are not directly downloadable in code, the function google_drive_file_id() extracts the unique file identifier from each URL using a regular expression (shortened as regex or regexp, which is a sequence of characters that forms a specific search pattern. It is used by string-searching algorithms to "find" or "find and replace" patterns in text, or to validate whether an input matches a specific set of rules.)

Here, the function searches for the /d/ portion of the URL and captures the file ID that follows it. If no valid file ID can be extracted, the function raises a ValueError to prevent the workflow from continuing with an invalid link.

The function download_drive_file() automates the downloading process. It takes a Google Drive URL and an output filename as inputs, extracts the file ID, constructs a downloadable link in the format https://drive.google.com/uc?id=FILE_ID, and uses the gdown library to download the file locally. The downloaded files are saved as ev_keepership.csv and ev_chargers.csv, and their paths are stored in variables for further reuse in the subsequent cells.

After downloading, both CSV files are read into pandas DataFrames using pd.read_csv(). The encoding parameter utf-8-sig ('UTF-8' stands for '8-Bit UCS Transformation Format' and represents the most widespread character encoding on the World Wide Web (WWW), sig" is the abbreviation of "signature" (i.e. signature utf-8 file).) is specified to correctly handle files containing a UTF-8 byte order mark (BOM), which is common in CSV files exported from spreadsheet software such as Excel. This avoids potential issues with corrupted column names or unexpected encoding artefacts.

Following that, the code then prints the dimensions of each dataset using the .shape attribute, which reports the number of rows and columns. This serves as an initial validation step to confirm that the files were loaded successfully. The first two rows of each DataFrame are then displayed using .head(2) to allow a quick inspection of the structure, column names, and example records before further preprocessing or analysis begins


In [ ]:
KEEPERSHIP_URL = "https://drive.google.com/file/d/1o5kQicaRDBe8xP_uQMOZ9cBe2TU-r-HM/view?usp=sharing"
CHARGER_URL = "https://drive.google.com/file/d/1-HjLsZAvcDyYHgn2jRwIVkiXV3102E6p/view?usp=sharing"


def google_drive_file_id(url: str) -> str:
    m = re.search(r"/d/([^/]+)", url)
    if not m:
        raise ValueError(f"Could not extract a Google Drive file id from: {url}")
    return m.group(1)

def download_drive_file(url: str, output_name: str) -> str:
    file_id = google_drive_file_id(url)
    download_url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(download_url, output_name, quiet=False, fuzzy=True)
    return output_name

keepership_path = download_drive_file(KEEPERSHIP_URL, "ev_keepership.csv")
charger_path = download_drive_file(CHARGER_URL, "ev_chargers.csv")


keepership_raw = pd.read_csv(keepership_path, encoding="utf-8-sig")
charger_raw = pd.read_csv(charger_path, encoding="utf-8-sig")


print("EV keepership:", keepership_raw.shape)
print("EV chargers:", charger_raw.shape)

display(keepership_raw.head(2), charger_raw.head(2))


# Raw Data Exploration

In [ ]:
print("Descriptive statistics for keepership_raw:")
display(keepership_raw.describe())

#`charger_raw` Data

In [ ]:
print("Descriptive statistics for charger_raw:")
display(charger_raw.describe())

# Common cleaning helpers

In [ ]:
WELSH_LAD_NAMES = {
    "isle of anglesey", "gwynedd", "conwy", "denbighshire", "flintshire", "wrexham",
    "powys", "ceredigion", "pembrokeshire", "carmarthenshire", "swansea", "neath port talbot",
    "bridgend", "vale of glamorgan", "cardiff", "rhondda cynon taf", "merthyr tydfil",
    "caerphilly", "blaenau gwent", "torfaen", "monmouthshire", "newport"
}

WELSH_LAD_NORMALISATION = {
    "sir ynys mon": "isle of anglesey", "ynys mon": "isle of anglesey", "isle of anglesey": "isle of anglesey",
    "sir y fflint": "flintshire", "caerdydd": "cardiff", "abertawe": "swansea", "casnewydd": "newport",
    "wrecsam": "wrexham", "rhondda cynon taff": "rhondda cynon taf", "the vale of glamorgan": "vale of glamorgan",
    "vale of glamorgan council": "vale of glamorgan", "cardiff council": "cardiff", "newport council": "newport",
}

# Welsh LAD code → canonical name. This prevents the charger code column `lad` being mistaken for a name.
WELSH_LAD_CODE_TO_NAME = {
    "W06000001": "isle of anglesey", "W06000002": "gwynedd", "W06000003": "conwy", "W06000004": "denbighshire",
    "W06000005": "flintshire", "W06000006": "wrexham", "W06000008": "ceredigion", "W06000009": "pembrokeshire",
    "W06000010": "carmarthenshire", "W06000011": "swansea", "W06000012": "neath port talbot", "W06000013": "bridgend",
    "W06000014": "vale of glamorgan", "W06000015": "cardiff", "W06000016": "rhondda cynon taf", "W06000018": "caerphilly",
    "W06000019": "blaenau gwent", "W06000020": "torfaen", "W06000021": "monmouthshire", "W06000022": "newport",
    "W06000023": "powys", "W06000024": "merthyr tydfil",
}

WELSH_LAD_NAME_TO_CODE = {v: k for k, v in WELSH_LAD_CODE_TO_NAME.items()}

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    # Drop completely empty unnamed columns that appear in the charger CSV.
    empty_unnamed = [c for c in out.columns if c.startswith("unnamed") and out[c].isna().all()]
    return out.drop(columns=empty_unnamed, errors="ignore")

def normalise_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text)
    text = text.replace("&", "and")
    text = re.sub(r"\b(county borough|county|city and county|council|local authority)\b", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return WELSH_LAD_NORMALISATION.get(text, text)

def normalise_lad_from_code_or_name(value) -> str:
    if pd.isna(value):
        return ""
    raw = str(value).strip().upper()
    if raw in WELSH_LAD_CODE_TO_NAME:
        return WELSH_LAD_CODE_TO_NAME[raw]
    return normalise_text(value)

def safe_uri_part(value) -> str:
    text = normalise_lad_from_code_or_name(value)
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return quote(text or "unknown")


def uri_fragment(value) -> str:
    """Create a readable URI fragment, e.g. Cardiff or Vale_of_Glamorgan."""
    text = normalise_lad_from_code_or_name(value)
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    if not text:
        return "Unknown"
    return quote("_".join(part.capitalize() for part in text.split("_")))

def first_existing_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def looks_like_lad_code_series(s: pd.Series) -> bool:
    vals = s.dropna().astype(str).str.strip()
    if vals.empty:
        return False
    return vals.str.match(r"^[A-Z]?\d{8}$|^W\d{8}$", case=False).mean() > 0.5

def welsh_lad_hit_count(s: pd.Series) -> int:
    return int(s.map(normalise_lad_from_code_or_name).isin(WELSH_LAD_NAMES).sum())

def detect_lad_name_col(df: pd.DataFrame, label: str):
    # Prefer actual name columns. Do not pick a column literally called `lad` when it contains W060000xx codes.
    name_candidates = [
        "lad_name", "lad name", "local authority name", "local authority district name",
        "local authority", "local authority district", "ons geography", "geography name",
        "geography", "area name", "area", "name", "la name"
    ]
    for c in name_candidates:
        if c in df.columns and not looks_like_lad_code_series(df[c]) and welsh_lad_hit_count(df[c]) > 0:
            return c

    best_col, best_hits = None, 0
    for c in df.columns:
        if df[c].dtype == "object" and not looks_like_lad_code_series(df[c]):
            hits = welsh_lad_hit_count(df[c])
            if hits > best_hits:
                best_col, best_hits = c, hits
    if best_col and best_hits > 0:
        return best_col
    raise ValueError(f"Could not detect LAD name column in {label}. Columns: {df.columns.tolist()}")

def detect_lad_code_col(df: pd.DataFrame):
    candidates = [
        "lad_code", "lad code", "lad", "ons code", "local authority district code",
        "local authority code", "area code", "geography code", "code", "la code"
    ]
    for c in candidates:
        if c in df.columns and looks_like_lad_code_series(df[c]):
            return c
    for c in df.columns:
        if looks_like_lad_code_series(df[c]):
            return c
    return first_existing_col(df, [c for c in candidates if c in df.columns])

def filter_wales_lads(df: pd.DataFrame, lad_name_col: str) -> pd.DataFrame:
    out = df.copy()
    out["lad_name"] = out[lad_name_col].map(normalise_lad_from_code_or_name)
    return out[out["lad_name"].isin(WELSH_LAD_NAMES)].copy()

def coerce_number(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
        errors="coerce"
    )


# Standardise EV keepership to LAD-year

This step prepares the EV keepership dataset by cleaning, reshaping, filtering, and standardising it. First, clean_columns() is run on keepership_raw to make column names consistent and easier to use. The code then looks for the local authority name and code columns with first_existing_col(), checking different possible names since source datasets often use various headings. If it cannot find a local authority name column, it uses detect_lad_name_col() as a backup.

Next, the code finds all quarterly keepership columns using a regular expression. It looks for column names like 2025q4, where the first four digits are the year and q1 to q4 show the quarter. If it does not find any quarterly columns, the code raises a ValueError since it cannot reshape the dataset without them.

The dataset is then changed from wide format to long format using melt(). In the wide format, each quarter has its own column. After melting, each row shows one local authority, one quarter, and one keepership value. This long format makes it easier to filter, group, and connect with other datasets.

If there is a fuel column, the code keeps only records for electric vehicles by looking for words like electric, battery, or plug. It then gets the year from the quarter column, changes the keepership values to numbers, standardises local authority names, and creates a local authority code. If there is no official code column, it makes a safe URI-friendly identifier from the local authority name.

The code then filters the data to keep only Welsh local authority districts. It does this by checking if the standardised local authority name is in WELSH_LAD_NAMES. This step makes sure that later analysis looks only at Wales.

Finally, the code picks the latest quarter available for each local authority in each year. This matters because EV keepership values are stock figures taken at certain times, not quarterly totals to be added up. For example, adding Q1, Q2, Q3, and Q4 would count the same vehicles more than once. The final standardised dataset, keepership_std, has one row for each local authority and year, with columns for local authority code, name, year, keepership value, and a URI-friendly identifier. The last two lines show the size of the dataset and print its first few rows for review.

In [ ]:
keepership = clean_columns(keepership_raw)

k_lad_name_col = first_existing_col(keepership, ["ons geography", "lad name", "local authority", "area name", "geography"])
k_lad_code_col = first_existing_col(keepership, ["ons code", "lad code", "area code", "geography code"])
if k_lad_name_col is None:
    k_lad_name_col = detect_lad_name_col(keepership, "EV keepership")

quarter_cols = [c for c in keepership.columns if re.match(r"^\d{4}q[1-4]$", c)]
if not quarter_cols:
    raise ValueError("No keepership quarter columns found. Expected columns such as 2025q4.")

id_vars = [c for c in [k_lad_code_col, k_lad_name_col, "fuel", "keepership"] if c and c in keepership.columns]
keepership_long = keepership.melt(
    id_vars=id_vars,
    value_vars=quarter_cols,
    var_name="quarter",
    value_name="keepership_value"
)

# Keep electric vehicles only when a fuel column is available.
if "fuel" in keepership_long.columns:
    keepership_long = keepership_long[
        keepership_long["fuel"].astype(str).str.lower().str.contains("electric|battery|plug", na=False)
    ].copy()

keepership_long["year"] = keepership_long["quarter"].str.extract(r"(\d{4})").astype(int)
keepership_long["keepership_value"] = coerce_number(keepership_long["keepership_value"])
keepership_long["lad_name"] = keepership_long[k_lad_name_col].map(normalise_text)
keepership_long["lad_code"] = keepership_long[k_lad_code_col].astype(str).str.strip() if k_lad_code_col else keepership_long["lad_name"].map(safe_uri_part)
keepership_long = keepership_long[keepership_long["lad_name"].isin(WELSH_LAD_NAMES)].copy()

# Use the latest quarter in each year per LAD rather than summing quarters, because quarterly stock values are repeated snapshots.
keepership_long["quarter_num"] = keepership_long["quarter"].str.extract(r"q([1-4])").astype(int)
keepership_std = (
    keepership_long
    .dropna(subset=["year", "lad_name", "keepership_value"])
    .sort_values(["lad_code", "lad_name", "year", "quarter_num"])
    .groupby(["lad_code", "lad_name", "year"], as_index=False)
    .tail(1)[["lad_code", "lad_name", "year", "keepership_value"]]
    .rename(columns={"keepership_value": "keepership"})
)
keepership_std["lad_uri_id"] = keepership_std["lad_name"].map(uri_fragment)

print("Standardised keepership rows:", keepership_std.shape)
display(keepership_std.head())


# Standardise EV charger counts to LAD-year

This cell cleans, reshapes, filters, and standardises the EV charger dataset. First, `clean_columns()` is applied to `charger_raw` to make the column names consistent. The code then detects the local authority district name and code columns using `detect_lad_name_col()` and `detect_lad_code_col()`, and prints the detected columns so they can be checked before continuing.

The code then searches for monthly charger count columns with names such as `jan-23`, `feb-2024`, or `dec-25`. If such columns exist, the dataset is treated as wide-format data, where each month is stored in a separate column. The code converts this into long format using `melt()`, so each row represents one local authority, one month, and one charger count value. The month labels are converted into real dates using `pd.to_datetime()`, first trying the short year format such as `Jan-23`, then falling back to the four-digit year format such as `Jan-2023`.

If no monthly columns are found, the code assumes the dataset may already be in long format. In that case, it tries to identify an existing `year` column and a charger count column such as `ev_count`, `charger count`, `count`, or `value`. If these cannot be found, the code raises a `ValueError` because it cannot determine how to standardise the charger data.

After reshaping, the charger counts are converted into numeric values using `coerce_number()`. The local authority name is then normalised using `normalise_lad_from_code_or_name()`. If a local authority code is available, it is converted to uppercase and used to fill in any missing names through the `WELSH_LAD_CODE_TO_NAME` mapping. If no code column exists, the code attempts to derive the local authority code from the name using `WELSH_LAD_NAME_TO_CODE`; otherwise, it creates a safe identifier from the name.

The data is then filtered to keep only Welsh local authority districts by checking membership in `WELSH_LAD_NAMES`. This ensures that the resulting table is aligned with the Welsh geography used in the keepership dataset.

The final aggregation depends on the structure of the input data. If monthly dates are available, the code selects the latest month for each local authority in each year, because charger counts are stock values at a point in time rather than monthly values that should be summed. If the dataset was already in long annual format, the code groups by local authority and year and sums the charger counts.

A validation step checks whether the standardised charger table is empty. If it is empty, the code displays samples of the cleaned and reshaped data to help diagnose problems with local authority detection or filtering, then raises an error. Finally, the year is converted to an integer, a URI-friendly local authority identifier is added, and the size and first rows of the final `count_std` table are displayed for inspection.



In [ ]:
chargers = clean_columns(charger_raw)

c_lad_name_col = detect_lad_name_col(chargers, "EV chargers")
c_lad_code_col = detect_lad_code_col(chargers)
print("Detected charger LAD name column:", c_lad_name_col)
print("Detected charger LAD code column:", c_lad_code_col)

month_cols = [
    c for c in chargers.columns
    if re.match(r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)-\d{2,4}$", c)
]
if not month_cols:
    # fallback for already-long data
    year_col = first_existing_col(chargers, ["year"])
    value_col = first_existing_col(chargers, ["ev_count", "ev charger count", "charger count", "count", "value"])
    if not (year_col and value_col):
        raise ValueError("No charger month columns found and no long-format year/value columns detected.")
    count_long = chargers.rename(columns={year_col: "year", value_col: "ev_chargers"}).copy()
    count_long["ev_chargers"] = coerce_number(count_long["ev_chargers"])
else:
    id_vars = [c for c in [c_lad_code_col, c_lad_name_col] if c]
    count_long = chargers.melt(id_vars=id_vars, value_vars=month_cols, var_name="month", value_name="ev_chargers")
    count_long["month_date"] = pd.to_datetime(count_long["month"], format="%b-%y", errors="coerce")
    bad = count_long["month_date"].isna()
    count_long.loc[bad, "month_date"] = pd.to_datetime(count_long.loc[bad, "month"], format="%b-%Y", errors="coerce")
    count_long["year"] = count_long["month_date"].dt.year
    count_long["ev_chargers"] = coerce_number(count_long["ev_chargers"])

# Use name when available; otherwise map from W060000xx code.
if c_lad_name_col:
    count_long["lad_name"] = count_long[c_lad_name_col].map(normalise_lad_from_code_or_name)
else:
    count_long["lad_name"] = ""
if c_lad_code_col:
    count_long["lad_code"] = count_long[c_lad_code_col].astype(str).str.strip().str.upper()
    count_long.loc[count_long["lad_name"].eq(""), "lad_name"] = count_long.loc[count_long["lad_name"].eq(""), "lad_code"].map(WELSH_LAD_CODE_TO_NAME)
else:
    count_long["lad_code"] = count_long["lad_name"].map(WELSH_LAD_NAME_TO_CODE).fillna(count_long["lad_name"].map(safe_uri_part))

count_long = count_long[count_long["lad_name"].isin(WELSH_LAD_NAMES)].copy()

if "month_date" in count_long.columns:
    count_std = (
        count_long
        .dropna(subset=["year", "lad_name", "ev_chargers"])
        .sort_values(["lad_code", "lad_name", "year", "month_date"])
        .groupby(["lad_code", "lad_name", "year"], as_index=False)
        .tail(1)[["lad_code", "lad_name", "year", "ev_chargers"]]
    )
else:
    count_std = (
        count_long
        .dropna(subset=["year", "lad_name", "ev_chargers"])
        .groupby(["lad_code", "lad_name", "year"], as_index=False)["ev_chargers"]
        .sum()
    )

if count_std.empty:
    print("Charger debugging sample after cleaning:")
    display(chargers.head())
    display(count_long.head())
    raise ValueError("Standardised charger table is empty. Check LAD name/code detection above.")

count_std["year"] = count_std["year"].astype(int)
count_std["lad_uri_id"] = count_std["lad_name"].map(uri_fragment)

print("Standardised charger rows:", count_std.shape)
display(count_std.head())


# Monthly and quarterly EV charger forecasting pipeline

This section increases the charger modelling sample size by using the charger dataset at its native monthly resolution instead of collapsing it immediately to annual LAD-year observations. It then derives a quarterly version from the monthly panel, creates lagged and rolling charger features, evaluates walk-forward forecasting performance, and produces recursive EV charger forecasts to 2045.

The annual model has only about `22 × 7 = 154` LAD-year observations. The monthly pipeline should provide many more LAD-time observations, while the quarterly pipeline provides an intermediate compromise between data volume and noise reduction.


In [ ]:
# ============================================================
# Monthly charger panel from charger_raw
# ============================================================

chargers_monthly_source = clean_columns(charger_raw)

c_lad_name_col_m = detect_lad_name_col(chargers_monthly_source, "EV chargers monthly")
c_lad_code_col_m = detect_lad_code_col(chargers_monthly_source)

month_cols_m = [
    c for c in chargers_monthly_source.columns
    if re.match(r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)-\d{2,4}$", c)
]

if not month_cols_m:
    raise ValueError(
        "No monthly charger columns were found. Expected columns such as jan-23, feb-2024, or dec-25."
    )

id_vars_m = [c for c in [c_lad_code_col_m, c_lad_name_col_m] if c]
charger_monthly_long = chargers_monthly_source.melt(
    id_vars=id_vars_m,
    value_vars=month_cols_m,
    var_name="month_label",
    value_name="ev_chargers"
)

charger_monthly_long["month_date"] = pd.to_datetime(
    charger_monthly_long["month_label"],
    format="%b-%y",
    errors="coerce"
)

bad_months = charger_monthly_long["month_date"].isna()
charger_monthly_long.loc[bad_months, "month_date"] = pd.to_datetime(
    charger_monthly_long.loc[bad_months, "month_label"],
    format="%b-%Y",
    errors="coerce"
)

charger_monthly_long["ev_chargers"] = coerce_number(charger_monthly_long["ev_chargers"])

# Use name when available; otherwise map Welsh LAD code to name.
if c_lad_name_col_m:
    charger_monthly_long["lad_name"] = charger_monthly_long[c_lad_name_col_m].map(normalise_lad_from_code_or_name)
else:
    charger_monthly_long["lad_name"] = ""

if c_lad_code_col_m:
    charger_monthly_long["lad_code"] = charger_monthly_long[c_lad_code_col_m].astype(str).str.strip().str.upper()
    missing_names = charger_monthly_long["lad_name"].eq("")
    charger_monthly_long.loc[missing_names, "lad_name"] = charger_monthly_long.loc[missing_names, "lad_code"].map(WELSH_LAD_CODE_TO_NAME)
else:
    charger_monthly_long["lad_code"] = charger_monthly_long["lad_name"].map(WELSH_LAD_NAME_TO_CODE)

charger_monthly_std = (
    charger_monthly_long
    .dropna(subset=["month_date", "lad_name", "ev_chargers"])
    .loc[lambda d: d["lad_name"].isin(WELSH_LAD_NAMES)]
    .copy()
)

charger_monthly_std["lad_code"] = charger_monthly_std["lad_code"].fillna(
    charger_monthly_std["lad_name"].map(WELSH_LAD_NAME_TO_CODE)
)
charger_monthly_std["lad_uri_id"] = charger_monthly_std["lad_code"].fillna(
    charger_monthly_std["lad_name"].map(safe_uri_part)
)
charger_monthly_std["year"] = charger_monthly_std["month_date"].dt.year.astype(int)
charger_monthly_std["month"] = charger_monthly_std["month_date"].dt.month.astype(int)
charger_monthly_std["quarter"] = charger_monthly_std["month_date"].dt.quarter.astype(int)
charger_monthly_std["period"] = charger_monthly_std["month_date"].dt.to_period("M").astype(str)

# Keep one value per LAD-month. Charger counts are stock observations, so duplicates are averaged defensively.
charger_monthly_std = (
    charger_monthly_std
    .groupby(["lad_code", "lad_uri_id", "lad_name", "month_date", "year", "month", "quarter", "period"], as_index=False)["ev_chargers"]
    .mean()
    .sort_values(["lad_name", "month_date"])
    .reset_index(drop=True)
)

charger_monthly_std["ev_chargers"] = charger_monthly_std["ev_chargers"].round().astype(int)

print("Monthly charger panel shape:", charger_monthly_std.shape)
print("LAD count:", charger_monthly_std["lad_name"].nunique())
print("Date range:", charger_monthly_std["month_date"].min().date(), "-", charger_monthly_std["month_date"].max().date())
display(charger_monthly_std.head(20))

charger_monthly_std.to_csv("cleets_ev_chargers_monthly_panel.csv", index=False)
print("Saved: cleets_ev_chargers_monthly_panel.csv")


In [ ]:
# ============================================================
# Quarterly charger panel derived from monthly charger counts
# ============================================================

charger_quarterly_std = charger_monthly_std.copy()
charger_quarterly_std["quarter_period"] = charger_quarterly_std["month_date"].dt.to_period("Q")

# Charger counts are stock values, so take the latest month available within each LAD-quarter.
charger_quarterly_std = (
    charger_quarterly_std
    .sort_values(["lad_name", "quarter_period", "month_date"])
    .groupby(["lad_code", "lad_uri_id", "lad_name", "quarter_period"], as_index=False)
    .tail(1)
    .copy()
)

charger_quarterly_std["quarter_start"] = charger_quarterly_std["quarter_period"].dt.start_time
charger_quarterly_std["quarter_end"] = charger_quarterly_std["quarter_period"].dt.end_time.dt.normalize()
charger_quarterly_std["year"] = charger_quarterly_std["quarter_period"].dt.year.astype(int)
charger_quarterly_std["quarter"] = charger_quarterly_std["quarter_period"].dt.quarter.astype(int)
charger_quarterly_std["period"] = charger_quarterly_std["quarter_period"].astype(str)

charger_quarterly_std = (
    charger_quarterly_std[[
        "lad_code", "lad_uri_id", "lad_name", "quarter_period", "quarter_start", "quarter_end",
        "year", "quarter", "period", "ev_chargers"
    ]]
    .sort_values(["lad_name", "quarter_start"])
    .reset_index(drop=True)
)

print("Quarterly charger panel shape:", charger_quarterly_std.shape)
print("LAD count:", charger_quarterly_std["lad_name"].nunique())
print("Quarter range:", charger_quarterly_std["period"].min(), "-", charger_quarterly_std["period"].max())
display(charger_quarterly_std.head(20))

charger_quarterly_std.to_csv("cleets_ev_chargers_quarterly_panel.csv", index=False)
print("Saved: cleets_ev_chargers_quarterly_panel.csv")


In [ ]:
# ============================================================
# Feature engineering, walk-forward evaluation, and recursive forecast
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import inspect
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def rmse(y_true, y_pred):
    """Root mean squared error (the squared=False argument was removed in scikit-learn 1.6)."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def add_charger_time_features(df, frequency="M", allow_missing_target=False):
    """Create lagged and rolling charger features for monthly or quarterly LAD panels.

    allow_missing_target=True keeps rows whose ev_chargers is still unknown (the periods to be
    forecast); their features are built from the observed and previously predicted history.
    All features use values strictly before the current period, so nothing leaks from the target.
    """
    out = df.copy().sort_values(["lad_name", "date"]).reset_index(drop=True)

    if frequency == "M":
        lags = [1, 2, 3, 6, 12]
        rolling_windows = [3, 6, 12]
        out["season"] = out["date"].dt.month.astype(int)
        out["season_sin"] = np.sin(2 * np.pi * out["season"] / 12)
        out["season_cos"] = np.cos(2 * np.pi * out["season"] / 12)
    elif frequency == "Q":
        lags = [1, 2, 4]
        rolling_windows = [2, 4]
        out["season"] = out["date"].dt.quarter.astype(int)
        out["season_sin"] = np.sin(2 * np.pi * out["season"] / 4)
        out["season_cos"] = np.cos(2 * np.pi * out["season"] / 4)
    else:
        raise ValueError("frequency must be 'M' or 'Q'.")

    min_date = out["date"].min()
    out["time_idx"] = ((out["date"].dt.year - min_date.year) * 12 + (out["date"].dt.month - min_date.month)).astype(int)
    if frequency == "Q":
        out["time_idx"] = (out["time_idx"] // 3).astype(int)

    g = out.groupby("lad_name", group_keys=False)
    for lag in lags:
        out[f"lag_{lag}"] = g["ev_chargers"].shift(lag)

    for window in rolling_windows:
        shifted = g["ev_chargers"].shift(1)
        out[f"roll_mean_{window}"] = shifted.groupby(out["lad_name"]).rolling(window).mean().reset_index(level=0, drop=True)
        out[f"roll_std_{window}"] = shifted.groupby(out["lad_name"]).rolling(window).std().reset_index(level=0, drop=True)

    # Growth of the previous period (lag_1 over lag_2). The earlier version divided the
    # current value by lag_1, which put the target itself into the features (leakage) and
    # made the rows to be forecast unusable because their target is unknown.
    out["growth_1"] = out["lag_1"] / out["lag_2"].replace(0, np.nan) - 1
    out["growth_1"] = out["growth_1"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    feature_cols = [
        "lad_name", "time_idx", "year", "season", "season_sin", "season_cos",
        *[f"lag_{lag}" for lag in lags],
        *[f"roll_mean_{window}" for window in rolling_windows],
        *[f"roll_std_{window}" for window in rolling_windows],
        "growth_1"
    ]

    out = out.dropna(subset=feature_cols).copy()
    if not allow_missing_target:
        out = out.dropna(subset=["ev_chargers"]).copy()
    return out, feature_cols


def make_one_hot_encoder():
    """Create a dense one-hot encoder compatible with older and newer scikit-learn versions."""
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_charger_model(model_name="hgb"):
    """Return a robust tabular forecasting model for small/medium charger panels."""
    if model_name == "rf":
        regressor = RandomForestRegressor(
            n_estimators=600,
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1
        )
    else:
        regressor = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.04,
            max_iter=500,
            l2_regularization=0.05,
            random_state=SEED
        )

    return Pipeline([
        ("encode", ColumnTransformer(
            transformers=[
                ("lad", make_one_hot_encoder(), ["lad_name"])
            ],
            remainder="passthrough",
            sparse_threshold=0.0
        )),
        ("model", regressor)
    ])


def walk_forward_charger_eval(panel, frequency="M", model_name="hgb", test_periods=6):
    """Evaluate charger forecasts using the last N observed periods as walk-forward tests."""
    model_df, feature_cols = add_charger_time_features(panel, frequency=frequency)
    unique_dates = sorted(model_df["date"].unique())
    test_dates = unique_dates[-test_periods:]

    rows = []
    for test_date in test_dates:
        train = model_df[model_df["date"] < test_date].copy()
        test = model_df[model_df["date"] == test_date].copy()

        if train.empty or test.empty:
            continue

        model = make_charger_model(model_name=model_name)
        model.fit(train[feature_cols], np.log1p(train["ev_chargers"]))
        pred = np.expm1(model.predict(test[feature_cols]))
        pred = np.clip(pred, 0, None)

        for (_, r), yhat in zip(test.iterrows(), pred):
            rows.append({
                "frequency": frequency,
                "model": model_name,
                "date": test_date,
                "lad_name": r["lad_name"],
                "actual_ev_chargers": float(r["ev_chargers"]),
                "predicted_ev_chargers": float(yhat)
            })

    pred_df = pd.DataFrame(rows)
    if pred_df.empty:
        raise ValueError("Walk-forward evaluation produced no predictions. Check available periods and lag settings.")

    metrics = []
    for date, part in pred_df.groupby("date"):
        y = part["actual_ev_chargers"].values
        yhat = part["predicted_ev_chargers"].values
        metrics.append({
            "frequency": frequency,
            "model": model_name,
            "date": date,
            "MAE": mean_absolute_error(y, yhat),
            "RMSE": rmse(y, yhat),
            "R2": r2_score(y, yhat) if len(np.unique(y)) > 1 else np.nan,
            "n": len(part)
        })

    return pred_df, pd.DataFrame(metrics)


def recursive_charger_forecast(panel, frequency="M", model_name="hgb", forecast_end="2045-12-31"):
    """Recursive multi-step forecast to 2045 for each LAD."""
    history = panel.copy().sort_values(["lad_name", "date"]).reset_index(drop=True)
    model_df, feature_cols = add_charger_time_features(history, frequency=frequency)

    model = make_charger_model(model_name=model_name)
    model.fit(model_df[feature_cols], np.log1p(model_df["ev_chargers"]))

    last_date = history["date"].max()
    if frequency == "M":
        future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), forecast_end, freq="MS")
    else:
        next_q = (last_date.to_period("Q") + 1).start_time
        future_dates = pd.period_range(next_q, forecast_end, freq="Q").to_timestamp(how="start")

    lad_meta = history[["lad_code", "lad_uri_id", "lad_name"]].drop_duplicates()
    forecasts = []

    for future_date in future_dates:
        future_rows = []
        for _, lad in lad_meta.iterrows():
            future_rows.append({
                "lad_code": lad["lad_code"],
                "lad_uri_id": lad["lad_uri_id"],
                "lad_name": lad["lad_name"],
                "date": future_date,
                "year": int(future_date.year),
                "ev_chargers": np.nan
            })

        candidate = pd.concat([history, pd.DataFrame(future_rows)], ignore_index=True)
        candidate_features, _ = add_charger_time_features(candidate, frequency=frequency, allow_missing_target=True)
        pred_rows = candidate_features[candidate_features["date"] == future_date].copy()
        if pred_rows.empty:
            raise RuntimeError(f"No feature rows could be built for {future_date.date()}; check the lag windows against the panel length.")

        pred = np.expm1(model.predict(pred_rows[feature_cols]))
        pred = np.clip(pred, 0, None)
        pred_rows["predicted_ev_chargers"] = pred

        append_rows = pred_rows[["lad_code", "lad_uri_id", "lad_name", "date", "year", "predicted_ev_chargers"]].copy()
        append_rows["ev_chargers"] = append_rows["predicted_ev_chargers"].round().astype(int)
        forecasts.append(append_rows.drop(columns=["ev_chargers"]).copy())

        # Feed predictions back as future history for the next recursive step.
        history = pd.concat([
            history,
            append_rows[["lad_code", "lad_uri_id", "lad_name", "date", "year", "ev_chargers"]]
        ], ignore_index=True)

    forecast_df = pd.concat(forecasts, ignore_index=True)
    forecast_df["frequency"] = frequency
    forecast_df["model"] = model_name
    return forecast_df


# Prepare monthly and quarterly modelling panels.
monthly_panel = charger_monthly_std.rename(columns={"month_date": "date"})[
    ["lad_code", "lad_uri_id", "lad_name", "date", "year", "ev_chargers"]
].copy()
monthly_panel["date"] = pd.to_datetime(monthly_panel["date"]).dt.to_period("M").dt.to_timestamp()

quarterly_panel = charger_quarterly_std.rename(columns={"quarter_start": "date"})[
    ["lad_code", "lad_uri_id", "lad_name", "date", "year", "ev_chargers"]
].copy()
quarterly_panel["date"] = pd.to_datetime(quarterly_panel["date"]).dt.to_period("Q").dt.to_timestamp(how="start")

monthly_backtest, monthly_metrics = walk_forward_charger_eval(
    monthly_panel,
    frequency="M",
    model_name="hgb",
    test_periods=min(12, monthly_panel["date"].nunique() // 4)
)

quarterly_backtest, quarterly_metrics = walk_forward_charger_eval(
    quarterly_panel,
    frequency="Q",
    model_name="hgb",
    test_periods=min(6, quarterly_panel["date"].nunique() // 4)
)

charger_frequency_metrics = pd.concat([monthly_metrics, quarterly_metrics], ignore_index=True)
charger_frequency_predictions = pd.concat([monthly_backtest, quarterly_backtest], ignore_index=True)

print("Walk-forward metrics by period:")
display(charger_frequency_metrics)

print("Mean walk-forward metrics by frequency:")
display(
    charger_frequency_metrics
    .groupby(["frequency", "model"], as_index=False)[["MAE", "RMSE", "R2"]]
    .mean()
    .sort_values("RMSE")
)

charger_frequency_predictions.to_csv("cleets_ev_chargers_monthly_quarterly_backtest_predictions.csv", index=False)
charger_frequency_metrics.to_csv("cleets_ev_chargers_monthly_quarterly_backtest_metrics.csv", index=False)
print("Saved backtest predictions and metrics.")

monthly_forecast_2045 = recursive_charger_forecast(
    monthly_panel,
    frequency="M",
    model_name="hgb",
    forecast_end="2045-12-31"
)
quarterly_forecast_2045 = recursive_charger_forecast(
    quarterly_panel,
    frequency="Q",
    model_name="hgb",
    forecast_end="2045-12-31"
)

monthly_forecast_2045.to_csv("cleets_ev_chargers_monthly_forecast_to_2045.csv", index=False)
quarterly_forecast_2045.to_csv("cleets_ev_chargers_quarterly_forecast_to_2045.csv", index=False)

print("Monthly forecast rows:", len(monthly_forecast_2045))
print("Quarterly forecast rows:", len(quarterly_forecast_2045))
print("Saved monthly and quarterly EV charger forecasts to 2045.")

display(monthly_forecast_2045.head(20))
display(quarterly_forecast_2045.head(20))


In [ ]:
# ============================================================
# Visual comparison: observed + monthly / quarterly forecasts
# ============================================================

monthly_plot = pd.concat([
    monthly_panel.assign(source="Observed", predicted_ev_chargers=monthly_panel["ev_chargers"])[["date", "source", "predicted_ev_chargers"]],
    monthly_forecast_2045.assign(source="Monthly forecast")[["date", "source", "predicted_ev_chargers"]]
], ignore_index=True)

quarterly_plot = pd.concat([
    quarterly_panel.assign(source="Observed", predicted_ev_chargers=quarterly_panel["ev_chargers"])[["date", "source", "predicted_ev_chargers"]],
    quarterly_forecast_2045.assign(source="Quarterly forecast")[["date", "source", "predicted_ev_chargers"]]
], ignore_index=True)

monthly_total = monthly_plot.groupby(["date", "source"], as_index=False)["predicted_ev_chargers"].sum()
quarterly_total = quarterly_plot.groupby(["date", "source"], as_index=False)["predicted_ev_chargers"].sum()

fig_monthly_chargers = px.line(
    monthly_total,
    x="date",
    y="predicted_ev_chargers",
    color="source",
    markers=True,
    title="Observed and forecast public EV chargers across Welsh LADs: monthly pipeline",
    labels={"predicted_ev_chargers": "Total public EV chargers", "date": "Date"}
)
fig_monthly_chargers.show()

fig_quarterly_chargers = px.line(
    quarterly_total,
    x="date",
    y="predicted_ev_chargers",
    color="source",
    markers=True,
    title="Observed and forecast public EV chargers across Welsh LADs: quarterly pipeline",
    labels={"predicted_ev_chargers": "Total public EV chargers", "date": "Quarter"}
)
fig_quarterly_chargers.show()


#  Time series graph

This cell below combines, reshapes, aggregates, and visualises EV keepership and EV charger data to produce simplified trend graphs across Wales. First, the code merges the standardised EV keepership dataset (`keepership_std`) and EV charger dataset (`count_std`) using common identifiers including local authority code, local authority name, year, and URI identifier. An outer join is used so that records are retained even when one dataset contains observations that are missing in the other. Missing values in EV keepership or charger counts are then replaced with zero to avoid gaps or errors in later calculations and visualisations.

The merged dataset is labelled as `Observed` data by adding `scenario` and `source` columns, which makes it compatible with later plotting workflows that may also include forecast scenarios. The data is then reshaped from wide format into long format using `melt()`. In the original wide format, EV keepership and charger counts exist as separate columns; after transformation, both become values in a single `metric` column with corresponding counts stored in a `value` column. This long format simplifies plotting multiple variables in a consistent structure.

To reduce visual complexity, the code aggregates values across all Welsh local authority districts (LADs). Using `groupby()` and `sum()`, it calculates yearly totals for each combination of year, scenario, source, and metric, producing a single national-level time series rather than separate local authority lines. This makes the resulting graphs easier to interpret and avoids overcrowding caused by plotting many LADs simultaneously.

Finally, the code creates an interactive line chart using Plotly Express. Separate panels (facets) are created for EV keepership and EV charger metrics, allowing each to have its own y-axis scale for readability. Different scenarios are distinguished by colour, while data sources are represented using line styles. Markers are added to highlight yearly observations, and facet labels are cleaned to improve readability. The resulting figure provides a simplified overview of observed trends in EV keepership and charging infrastructure over time across Wales.


In [ ]:
# Merge keepership and charger data
plot_df = keepership_std.merge(
    count_std,
    on=["lad_code", "lad_name", "year", "lad_uri_id"],
    how="outer"
)

# Fill missing values
plot_df["keepership"] = plot_df["keepership"].fillna(0)
plot_df["ev_chargers"] = plot_df["ev_chargers"].fillna(0)

# Add labels for observed data
plot_df["scenario"] = "Observed"
plot_df["source"] = "Observed"

# Convert to long format for plotting
melted_plot_df = plot_df.melt(
    id_vars=["lad_code", "lad_name", "year", "scenario", "source"],
    value_vars=["keepership", "ev_chargers"],
    var_name="metric",
    value_name="value"
)

# Aggregate across all LADs
overall_melted_plot_df = melted_plot_df.groupby(
    ["year", "scenario", "source", "metric"],
    as_index=False
)["value"].sum()

# Plot
fig_simplified_overall = px.line(
    overall_melted_plot_df,
    x="year",
    y="value",
    color="scenario",
    line_dash="source",
    facet_row="metric",
    title="Overall EV Keepership and Chargers: Observed Data (Aggregated Across LADs)",
    markers=True,
    height=600,
    labels={
        "value": "Total Count",
        "metric": "Metric"
    }
)

fig_simplified_overall.for_each_annotation(
    lambda a: a.update(text=a.text.replace("metric=", ""))
)

fig_simplified_overall.update_yaxes(matches=None)

fig_simplified_overall.show()

# Side by side visualisation

This cell below prepares and visualises EV keepership and EV charger trends across all Welsh local authority districts (LADs). First, a copy of the merged dataset (`plot_df`) is created to preserve the original data while allowing modifications for plotting. The code then removes misleading zero-valued EV keepership observations for the year 2026 by replacing them with missing values (`None`). This step prevents the plotted keepership lines from artificially dropping to zero in 2026 when no valid observation exists, ensuring that the visualisation more accurately reflects the available data.

Next, the dataset is reshaped from wide format into long format using `melt()`. In the original structure, EV keepership and EV charger counts are stored as separate columns. After transformation, these become entries within a single `metric` column, while their corresponding numerical values are stored in a `value` column. This long-format representation is more suitable for plotting multiple variables within a single visualisation framework.

The code then creates an interactive line chart using Plotly Express to display trends for all Welsh LADs over time. Separate panels (facets) are created for EV keepership and EV charger counts so that both metrics can be viewed side by side while maintaining independent y-axis scales for readability. Each LAD is represented using a different colour, allowing trends to be compared across regions, while line styles distinguish between different data sources when available. Markers are added to highlight yearly observations, facet labels are cleaned for readability, and font sizes are adjusted to improve presentation quality.

The resulting figure provides an overview of temporal changes in EV ownership and charging infrastructure across all Welsh local authorities, while avoiding misleading visual artefacts caused by incomplete keepership data in 2026.


In [ ]:
import plotly.express as px

# Start from plot_df
all_lads_df = plot_df.copy()

# Remove only 2026 zero values for keepership
all_lads_df.loc[
    (all_lads_df["year"] == 2026) & (all_lads_df["keepership"] == 0),
    "keepership"
] = None

# Reshape to long format
all_lads_plot_data = all_lads_df.melt(
    id_vars=["lad_code", "lad_name", "year", "scenario", "source"],
    value_vars=["keepership", "ev_chargers"],
    var_name="metric",
    value_name="value"
)

# Plot all LADs
fig_all_lads = px.line(
    all_lads_plot_data,
    x="year",
    y="value",
    color="lad_name",
    line_dash="source",
    facet_col="metric",
    title="All Welsh LADs: EV Keepership and Chargers",
    markers=True,
    height=650,
    labels={
        "value": "Count",
        "metric": "Metric",
        "lad_name": "LAD Name"
    }
)

fig_all_lads.for_each_annotation(
    lambda a: a.update(text=a.text.replace("metric=", ""))
)

fig_all_lads.update_yaxes(matches=None)

fig_all_lads.update_layout(
    title_font=dict(size=18),
    font=dict(size=11),
    legend_title_text="LAD Name",
    legend_font_size=10
)

fig_all_lads.show()

# Building CLEETS ontology

This cell constructs and exports the CLEETS ontology as an RDF/OWL knowledge model for representing electric vehicle (EV), charger, local authority, deprivation, and temporal data in a semantically structured way. First, the code defines namespaces, including a custom `CLEETS` namespace for project-specific concepts and the W3C Time Ontology namespace (`TIME`) for representing temporal information. A new RDF graph is created and common semantic web vocabularies, including RDF, RDFS, OWL, and XSD, are bound to the graph to enable readable prefixes and standard semantic definitions.

The ontology itself is then declared and labelled as the **CLEETS ontology**. A set of domain-specific classes is created, including `EV`, `LAD`, `WelshLAD`, `Observation`, `EVKeepership`, `EVChargerCount`, `IncomeDeprivation`, and `Time`. These classes define the conceptual entities of the domain and provide a semantic structure for representing EV-related observations. Hierarchical relationships are also introduced using subclass definitions. For example, `WelshLAD` is defined as a subclass of `LAD`, while `EVKeepership`, `EVChargerCount`, and `IncomeDeprivation` are defined as subclasses of `Observation`, allowing specialised observations to inherit from a more general observation concept.

The code then defines object properties that describe relationships between entities. Specifically, `forLAD` links an observation to a local authority district and `forTime` links an observation to a time entity. Each property is defined with a domain and range to formally specify which classes may participate in the relationship. In addition, datatype properties are created to store literal values such as local authority names, codes, years, EV keepership counts, charger counts, deprivation values, and ratio-based indicators. These properties are assigned XML Schema datatypes (e.g., string, integer, float) to ensure consistent and interpretable data representation.

Finally, the code prints the total number of RDF triples in the ontology, serialises the ontology into a Turtle (`.ttl`) file, and downloads it locally. The Turtle file provides a portable, machine-readable representation of the ontology that can be reused in semantic web applications, linked data environments, knowledge graphs, or ontology reasoning workflows.             


In [ ]:
CLEETS = Namespace("http://w3id.org/def/cleets/")
TIME = Namespace("http://www.w3.org/2006/time#")

kg = Graph()
kg.bind("cleets", CLEETS)
kg.bind("time", TIME)
kg.bind("rdf", RDF)
kg.bind("rdfs", RDFS)
kg.bind("owl", OWL)
kg.bind("xsd", XSD)

kg.add((CLEETS[""], RDF.type, OWL.Ontology))
kg.add((CLEETS[""], RDFS.label, Literal("CLEETS ontology")))

classes = [
    "EV", "LAD", "WelshLAD", "Observation", "EVKeepership", "EVChargerCount",
    "IncomeDeprivation", "Time"
]
for cls in classes:
    kg.add((CLEETS[cls], RDF.type, OWL.Class))
    kg.add((CLEETS[cls], RDFS.label, Literal(cls)))

kg.add((CLEETS.WelshLAD, RDFS.subClassOf, CLEETS.LAD))
kg.add((CLEETS.EVKeepership, RDFS.subClassOf, CLEETS.Observation))
kg.add((CLEETS.EVChargerCount, RDFS.subClassOf, CLEETS.Observation))
kg.add((CLEETS.IncomeDeprivation, RDFS.subClassOf, CLEETS.Observation))
kg.add((CLEETS.Time, RDFS.subClassOf, TIME.TemporalEntity))

object_properties = {
    "forLAD": (CLEETS.Observation, CLEETS.LAD, "links an observation to a LAD"),
    "forTime": (CLEETS.Observation, CLEETS.Time, "links an observation to a time entity"),
}
for prop, (domain, range_, label) in object_properties.items():
    kg.add((CLEETS[prop], RDF.type, OWL.ObjectProperty))
    kg.add((CLEETS[prop], RDFS.domain, domain))
    kg.add((CLEETS[prop], RDFS.range, range_))
    kg.add((CLEETS[prop], RDFS.label, Literal(label)))

datatype_properties = {
    "ladName": XSD.string,
    "ladCode": XSD.string,
    "year": XSD.integer,
    "keepershipValue": XSD.float,
    "chargerCountValue": XSD.float,
    "incomeDeprivationValue": XSD.float,
    "keeperChargerRatio": XSD.float,
    "chargersPerKeeper": XSD.float,
}
for prop, dtype in datatype_properties.items():
    kg.add((CLEETS[prop], RDF.type, OWL.DatatypeProperty))
    kg.add((CLEETS[prop], RDFS.range, dtype))

print("Ontology triples:", len(kg))
kg.serialize("cleets_ontology.ttl", format="turtle")
files.download("cleets_ontology.ttl")


# Populate CLEETS ontology with the datasets:

This code cell populates CLEETS ontology with electric vehicle (EV) keepership and EV charger observations for Welsh local authority districts (LADs), and exports the completed graph in semantic web formats. Initially, a set of helper functions is defined to create consistent Uniform Resource Identifiers (URIs) for LADs, time entities, and observations. These functions generate unique identifiers for each local authority, year, EV keepership observation, and EV charger observation to make sure that entities are uniquely referenced and linked within the knowledge graph.

The add_lad() function creates LAD entities in the graph and assigns them both the LAD and WelshLAD classes, reflecting the ontology hierarchy defined earlier. It also attaches descriptive attributes such as local authority name and local authority code as datatype properties. Similarly, the add_time() function creates temporal entities for each year, classifies them as both CLEETS.Time and TIME.TemporalEntity, and records the associated year value. These helper functions reduce duplication and ensure consistency when adding LAD and time entities repeatedly.

The code then iterates through the standardised EV keepership dataset (keepership_std) and creates RDF observation instances for each local authority and year combination. For each record, it creates or retrieves the LAD and time entities, generates a unique EV keepership observation URI, assigns the observation to the EVKeepership class, and links it to both the corresponding LAD and year using the forLAD and forTime object properties. The observed keepership value is stored as a datatype property (keepershipValue) using a (float) numeric datatype.

A similar process is applied to the standardised EV charger dataset (count_std). For each local authority-year combination, the code creates an EVChargerCount observation, links it to the relevant LAD and temporal entity, and records the charger count value using the chargerCountValue datatype property. This approach results in a temporally structured knowledge graph in which EV-related observations are semantically linked to specific locations and years.

Finally, CLEETS knowledge graph is serialised and exported in two semantic web formats: Turtle (.ttl) and RDF/XML (.rdf). The total number of RDF triples in the graph is printed to provide a summary of graph size, and both files are downloaded locally for reuse, querying, sharing, or integration into linked data and knowledge graph workflows.

In [ ]:
def lad_uri(lad_uri_id):
    return CLEETS[f"LAD#{uri_fragment(lad_uri_id)}"]

def time_uri(year):
    return CLEETS[f"Time#{int(year)}"]

def keepership_obs_uri(lad_uri_id, year):
    return CLEETS[f"EVKeepership#{uri_fragment(lad_uri_id)}_{int(year)}"]

def charger_obs_uri(lad_uri_id, year):
    return CLEETS[f"EVChargerCount#{uri_fragment(lad_uri_id)}_{int(year)}"]

def add_lad(lad_uri_id, lad_name, lad_code=None):
    u = lad_uri(lad_uri_id)
    kg.add((u, RDF.type, CLEETS.LAD))
    kg.add((u, RDF.type, CLEETS.WelshLAD))
    kg.add((u, CLEETS.ladName, Literal(str(lad_name), datatype=XSD.string)))
    if lad_code is not None and str(lad_code).lower() != "nan":
        kg.add((u, CLEETS.ladCode, Literal(str(lad_code), datatype=XSD.string)))
    return u

def add_time(year):
    u = time_uri(year)
    kg.add((u, RDF.type, CLEETS.Time))
    kg.add((u, RDF.type, TIME.TemporalEntity))
    kg.add((u, CLEETS.year, Literal(int(year), datatype=XSD.integer)))
    return u

for _, row in keepership_std.iterrows():
    lad = add_lad(row["lad_uri_id"], row["lad_name"], row.get("lad_code"))
    t = add_time(row["year"])
    obs = keepership_obs_uri(row["lad_uri_id"], row["year"])
    kg.add((obs, RDF.type, CLEETS.EVKeepership))
    kg.add((obs, CLEETS.forLAD, lad))
    kg.add((obs, CLEETS.forTime, t))
    kg.add((obs, CLEETS.keepershipValue, Literal(float(row["keepership"]), datatype=XSD.float)))

for _, row in count_std.iterrows():
    lad = add_lad(row["lad_uri_id"], row["lad_name"], row.get("lad_code"))
    t = add_time(row["year"])
    obs = charger_obs_uri(row["lad_uri_id"], row["year"])
    kg.add((obs, RDF.type, CLEETS.EVChargerCount))
    kg.add((obs, CLEETS.forLAD, lad))
    kg.add((obs, CLEETS.forTime, t))
    kg.add((obs, CLEETS.chargerCountValue, Literal(float(row["ev_chargers"]), datatype=XSD.float)))

kg.serialize("cleets_three_dataset_temporal_kg.ttl", format="turtle")
kg.serialize("cleets_three_dataset_temporal_kg.rdf", format="xml")
print("Total triples after population:", len(kg))


files.download("cleets_three_dataset_temporal_kg.ttl")
files.download("cleets_three_dataset_temporal_kg.rdf")

##  Load CLEETS knowledge graph

This cell loads CLEETS knowledge graph (KG) into memory or reuses an existing RDFLib graph if one has already been created. First, the code checks whether the graph variable `kg` already exists in the notebook environment. If it is available, the existing graph is reused and the total number of RDF triples is printed. This avoids unnecessary reloading and improves efficiency when the knowledge graph has already been generated in a previous step.

If the graph variable does not exist, the code assumes that the notebook session has been restarted or that no graph has yet been loaded. In this case, it prompts the user to upload an RDF knowledge graph file using the Google Colab upload interface. A new RDFLib graph object is then created to store the uploaded semantic data.

To maximise compatibility, the code attempts to parse the uploaded file first as Turtle (`.ttl`) format and, if this fails, automatically retries using RDF/XML (`.rdf` or `.xml`) format. This flexible loading strategy allows the notebook to work with multiple standard semantic web serialisation formats without requiring manual configuration. Once loaded, the total number of RDF triples in the graph is printed as a validation step to confirm successful loading and provide an indication of graph size.

The resulting graph (`kg`) serves as the working temporal knowledge graph for subsequent semantic querying, reasoning, and analysis tasks.



In [ ]:
try:
    kg
    print("Using existing rdflib Graph variable: kg")
    print("Triples:", len(kg))
except NameError:
    from google.colab import files

    uploaded = files.upload()
    rdf_path = list(uploaded.keys())[0]

    kg = Graph()

    # Try Turtle first, then RDF/XML.
    try:
        kg.parse(rdf_path, format="turtle")
        print("Loaded RDF as Turtle:", rdf_path)
    except Exception:
        kg.parse(rdf_path, format="xml")
        print("Loaded RDF as RDF/XML:", rdf_path)

    print("Triples:", len(kg))


# Extract data (integer electric vehicle (EV) keepership and charger counts) using SPARQL query language

The cell below queries CLEETS knowledge graph using SPARQL to extract yearly electric vehicle (EV) keepership and EV charger counts for Welsh local authority districts (LADs), transforms the results into a structured tabular dataset, and exports the final panel data for analysis. The process converts semantically structured RDF knowledge graph data into a conventional dataframe suitable for statistical analysis, forecasting, or visualisation.

First, the SPARQL query is defined to retrieve EV keepership observations, EV charger observations, local authority information, and temporal information from the knowledge graph. The query searches for `EVKeepership` and `EVChargerCount` observations, links them to the same local authority and time entity using the `forLAD` and `forTime` relationships, and retrieves the associated numeric values (`keepershipValue` and `chargerCountValue`). It also restricts local authorities to those classified as `WelshLAD` and extracts the LAD name and year value. The query results are ordered by local authority name and year to produce a chronological time series.

The query is then executed using RDFLib’s `kg.query()` function. For each returned result, the code extracts the LAD URI, LAD name, year, EV keepership value, and EV charger count and stores them in a Python dictionary. Numeric values are converted to integers to standardise the representation and remove any unnecessary floating-point formatting. These records are accumulated in a list and converted into a pandas DataFrame named `panel_df`.

A validation step checks whether the query returned any data. If the dataframe is empty, the code raises an informative error indicating that the expected CLEETS ontology classes (`EVKeepership`, `EVChargerCount`, `WelshLAD`) or properties (`forLAD`, `forTime`, `keepershipValue`, `chargerCountValue`) may not exist in the knowledge graph. This prevents later analysis from continuing with invalid or incomplete graph data.

The dataframe is then filtered to retain only yearly Welsh LAD observations between 2019 and 2025. This ensures that the resulting panel contains a consistent temporal range for analysis. Duplicate records are removed defensively using local authority and year as identifiers, ensuring that only one observation per LAD-year combination remains. The data are sorted chronologically and reset into a clean tabular structure.

To improve consistency and avoid datatype ambiguity, the year, EV keepership, and EV charger columns are explicitly converted to integer types. Summary information is then printed, including the dataframe dimensions, number of LADs, temporal range, and column datatypes. A preview of the first rows is displayed to support verification of the extracted data.

Finally, the cleaned panel dataset is exported as a CSV file (`cleets_sparql_integer_lad_year_panel_2019_2025.csv`). This file provides a structured LAD-by-year panel extracted directly from the temporal knowledge graph and can be used for downstream statistical modelling, forecasting, machine learning, or visual analytics.


In [ ]:
sparql_query = """
PREFIX cleets: <http://w3id.org/def/cleets/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?lad ?ladName ?year ?keepership ?evChargers
WHERE {
  ?kObs rdf:type cleets:EVKeepership ;
        cleets:forLAD ?lad ;
        cleets:forTime ?time ;
        cleets:keepershipValue ?keepership .

  ?cObs rdf:type cleets:EVChargerCount ;
        cleets:forLAD ?lad ;
        cleets:forTime ?time ;
        cleets:chargerCountValue ?evChargers .

  ?lad rdf:type cleets:WelshLAD ;
       cleets:ladName ?ladName .

  ?time cleets:year ?year .
}
ORDER BY ?ladName ?year
"""

rows = []

for r in kg.query(sparql_query):
    rows.append({
        "lad_uri": str(r.lad),
        "lad_name": str(r.ladName).strip(),
        "year": int(round(float(r.year))),
        "ev_keepership": int(round(float(r.keepership))),
        "ev_chargers": int(round(float(r.evChargers)))
    })

panel_df = pd.DataFrame(rows)

if panel_df.empty:
    raise ValueError(
        "SPARQL query returned no rows. Check that the KG uses CLEETS classes "
        "EVKeepership, EVChargerCount, WelshLAD, and properties forLAD, forTime, "
        "keepershipValue, chargerCountValue."
    )

# Keep Welsh LAD yearly time-series from 2019 to 2025.
panel_df = panel_df[
    (panel_df["year"] >= 2011)
    & (panel_df["year"] <= 2025)
].copy()

# Remove duplicates defensively.
panel_df = (
    panel_df
    .drop_duplicates(subset=["lad_uri", "year"], keep="first")
    .sort_values(["lad_name", "year"])
    .reset_index(drop=True)
)

# Enforce integer dtypes.
panel_df["year"] = panel_df["year"].astype(int)
panel_df["ev_keepership"] = panel_df["ev_keepership"].astype(int)
panel_df["ev_chargers"] = panel_df["ev_chargers"].astype(int)

print("Extracted panel shape:", panel_df.shape)
print("LAD count:", panel_df["lad_name"].nunique())
print("Year range:", panel_df["year"].min(), "-", panel_df["year"].max())
print(panel_df.dtypes)

display(panel_df.head(30))

panel_df.to_csv("cleets_sparql_integer_lad_year_panel_2019_2025.csv", index=False)
print("Saved: cleets_sparql_integer_lad_year_panel_2019_2025.csv")


# Output data checks

This cell checks that the panel contains the expected 22 Welsh LADs and yearly data from 2019 to 2025.

Expected rows if fully balanced:

\[
22 \times 7 = 154
\]


In [ ]:
expected_years = list(range(2019, 2026))

coverage = (
    panel_df
    .groupby("lad_name")["year"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .reset_index()
)

coverage["n_years"] = coverage["year"].apply(len)
coverage["complete_2019_2025"] = coverage["year"].apply(lambda ys: ys == expected_years)

display(coverage)

print("Number of LADs:", panel_df["lad_name"].nunique())
print("Number of rows:", len(panel_df))
print("Complete LAD series:", coverage["complete_2019_2025"].sum(), "/", len(coverage))

if panel_df["lad_name"].nunique() != 22:
    print("WARNING: LAD count is not 22. Check LAD naming or KG population.")

if not coverage["complete_2019_2025"].all():
    print("WARNING: Some LADs do not have complete yearly coverage.")


# SPARQL query output validation

The below cell validates, cleans, and exports the SPARQL-extracted LAD-year panel dataset. It first checks that `panel_df` contains the required columns: LAD URI, LAD name, year, EV keepership, and EV charger count. If any required column is missing, the code stops with an informative error.

The code then keeps only the required columns and converts the year, keepership, and charger values into numeric format. Infinite values are replaced with missing values, and any rows containing missing, invalid, or negative values are identified. The number of invalid rows is printed, and any problematic rows are displayed for inspection.

After validation, invalid rows are removed. The numeric columns are rounded and converted to integers, duplicate LAD-year records are dropped, and the dataset is restricted to the 2019–2025 period. The cleaned panel is sorted by LAD name and year, and summary information is printed, including dataset shape, number of LADs, year range, and datatypes.

The cell also checks temporal coverage for each LAD. It lists the years available for each local authority, counts how many years are present, and flags whether each LAD has complete coverage from 2019 to 2025. Finally, the cleaned dataset is saved as `sparql_extracted_integer_panel_2019_2025.csv` for downstream modelling, forecasting, or visualisation.

In [ ]:
required_cols = [
    "lad_uri",
    "lad_name",
    "year",
    "ev_keepership",
    "ev_chargers"
]

missing = [c for c in required_cols if c not in panel_df.columns]

if missing:
    raise ValueError(f"panel_df is missing required columns: {missing}")

panel_df = panel_df[required_cols].copy()

panel_df["year"] = pd.to_numeric(panel_df["year"], errors="coerce")
panel_df["ev_keepership"] = pd.to_numeric(panel_df["ev_keepership"], errors="coerce")
panel_df["ev_chargers"] = pd.to_numeric(panel_df["ev_chargers"], errors="coerce")

panel_df = panel_df.replace([np.inf, -np.inf], np.nan)

invalid_rows = panel_df[
    panel_df[["year", "ev_keepership", "ev_chargers"]].isna().any(axis=1)
    | (panel_df["ev_keepership"] < 0)
    | (panel_df["ev_chargers"] < 0)
].copy()

print("Invalid rows:", len(invalid_rows))

if len(invalid_rows) > 0:
    display(invalid_rows)

panel_df = panel_df.dropna(subset=["year", "ev_keepership", "ev_chargers"]).copy()

panel_df = panel_df[
    (panel_df["ev_keepership"] >= 0)
    & (panel_df["ev_chargers"] >= 0)
].copy()

panel_df["year"] = panel_df["year"].round().astype(int)
panel_df["ev_keepership"] = panel_df["ev_keepership"].round().astype(int)
panel_df["ev_chargers"] = panel_df["ev_chargers"].round().astype(int)

panel_df = (
    panel_df
    .drop_duplicates(subset=["lad_uri", "year"], keep="first")
    .query("2011 <= year <= 2025")
    .sort_values(["lad_name", "year"])
    .reset_index(drop=True)
)

print("Panel shape:", panel_df.shape)
print("LAD count:", panel_df["lad_name"].nunique())
print("Year range:", panel_df["year"].min(), "-", panel_df["year"].max())
print(panel_df.dtypes)

display(panel_df.head(30))

coverage = (
    panel_df
    .groupby("lad_name")["year"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .reset_index(name="years")
)

coverage["n_years"] = coverage["years"].apply(len)
coverage["complete_2019_2025"] = coverage["years"].apply(lambda ys: ys == list(range(2019, 2026)))

display(coverage)

panel_df.to_csv("sparql_extracted_integer_panel_2019_2025.csv", index=False)

print("Saved: sparql_extracted_integer_panel_2019_2025.csv")



# Capacity Scenario table

When forecasting EV adoption to $2045$--$2045$, the capacity parameter $K$ in a bounded or logistic growth model represents the **long-run maximum level** that EV keepership or charger deployment is assumed to approach by the end of the forecast horizon. Rather than representing an immediate prediction, $K$ acts as a saturation level that constrains growth over time. Consequently, when a multiplier such as $10\times$ the latest observed value is used, the model implicitly assumes that EV keepership and charging infrastructure could eventually increase to approximately ten times their current observed level during the mature stages of adoption.

For example, Cardiff recorded $32{,}140$ EVs in December $2025$ and $309$ public EV chargers in January $2026$. Under a central scenario using a multiplier of $10$, the assumed carrying capacities become:

$$
K_{\text{EV keepership}}
=
32{,}140 \times 10
=
321{,}400
$$

$$
K_{\text{EV chargers}}
=
309 \times 10
=
3{,}090
$$

These values do not imply that Cardiff will immediately reach $321{,}400$ EVs or $3{,}090$ chargers. Instead, the model assumes that EV keepership and charging infrastructure may continue to grow gradually and approach these levels by approximately $2045$, depending on the estimated growth trajectory.

The challenge with this assumption is that the multiplier is **time-insensitive** and structurally simplistic. Whether the forecast ends in $2030$ or $2045$ the carrying capacity remains unchanged:

$$
321{,}400 \text{ EVs}
$$

$$
3{,}090 \text{ chargers}
$$

even though longer forecast horizons generally imply stronger policy intervention, increased EV affordability, improved charging infrastructure, technological maturity, and continued replacement of internal combustion engine vehicles. Consequently, assumptions about $K$ should ideally reflect plausible conditions near the end of the forecast period rather than rely solely on fixed proportional scaling.

A second issue concerns realism. If Cardiff’s future total registered vehicle stock were substantially lower than $321{,}400$ vehicles, then the forecast would imply more EVs than vehicles in existence, which would not be plausible. Likewise, the realism of $3{,}090$ chargers depends on expected transport demand and charger-to-vehicle ratios. For example, assuming one public charger for every $40$ EVs:

$$
\frac{321{,}400}{40}
\approx
8{,}035
\text{ chargers}
$$

This suggests that $3{,}090$ chargers may underestimate infrastructure requirements relative to the assumed EV saturation level.

Therefore, while multipliers such as:

$$
\text{low}=6\times,\qquad
\text{central}=10\times,\qquad
\text{high}=15\times
$$

are useful for exploratory scenario analysis, they should be interpreted as **placeholder assumptions** rather than policy-grounded long-run capacities for forecasts extending to $2045$--$2053$.

In [ ]:
latest_counts = (
    panel_df
    .sort_values("year")
    .groupby(["lad_uri", "lad_name"], as_index=False)
    .tail(1)
    [["lad_uri", "lad_name", "ev_keepership", "ev_chargers"]]
)

capacity_rows = []

# Placeholder scenario multipliers.
# Replace these with policy-grounded capacity assumptions where possible.
scenario_multipliers = {
    "low": 6,
    "central": 10,
    "high": 15
}

for _, row in latest_counts.iterrows():
    latest_ev = int(row["ev_keepership"])
    latest_chargers = int(row["ev_chargers"])

    for scenario, multiplier in scenario_multipliers.items():
        capacity_rows.append({
            "scenario": scenario,
            "lad_uri": row["lad_uri"],
            "lad_name": row["lad_name"],
            "latest_ev_keepership": latest_ev,
            "latest_ev_chargers": latest_chargers,

            # EDIT THESE TWO COLUMNS MANUALLY IF YOU HAVE BETTER LAD CAPACITY DATA
            "K_ev_keepership": max(latest_ev + 1, int(round(latest_ev * multiplier))),
            "K_ev_chargers": max(latest_chargers + 1, int(round(latest_chargers * multiplier)))
        })

capacity_df = pd.DataFrame(capacity_rows)

capacity_df.to_csv("editable_lad_capacity_scenarios.csv", index=False)

display(capacity_df)

print("Saved editable file: editable_lad_capacity_scenarios.csv")


In [ ]:
print('Current contents of editable_lad_capacity_scenarios.csv:')
display(capacity_df)

In [ ]:
print('Current editable capacity scenarios (in-cell editing enabled):')
display(capacity_df)

# Bounded logistic model

The below cell introduce the bounded logistic growth model for forecasting EV keepership and EV charger growth under an assumed  above maximum capacity (\(K\)). The model is designed to represent growth that accelerates initially and gradually slows as it approaches a predefined upper limit This makes it suitable for technologies such as EV adoption that are expected to diffuse over time and eventually stabilise.

The function `logistic_curve()` defines the mathematical form of the logistic growth model:

\[
f(t)=\frac{K}{1+\exp(-r(t-t_0))}
\]

where:

- \(K\) stands for the carrying capacity or long-run maximum level,
- \(r\) stands for the growth rate,
- \(t_0\) stands for the midpoint year (the period of fastest growth), and
- \(t\) is the time.

This formulation produces an S-shaped growth curve in which adoption begins slowly, accelerates during expansion, and stabilises as it nears saturation.

The function [fit_bounded_logistic()] estimates this bounded growth trajectory using observed yearly EV keepership or charger counts and an externally defined carrying capacity.

The function therefore accepts four inputs:

1. observed years,
2. observed values,
3. an assumed carrying capacity (\(K\)), and
4. forecast years.

A validation step checks that enough observations are available, since nonlinear growth models are unstable when fitted to very small samples (less than five observations).

The function also make sure that the carrying capacity exceeds the largest observed value. If the supplied \(K\) is too small, it is automatically increased to approximately **120\% of the maximum observed value** to prevent impossible growth behaviour. Observed values are also constrained to lie strictly within the interval:

\[
0 < \text{value} < K
\]

to avoid numerical instability during estimation.

For simple model fitting, the carrying capacity (\(K\)) is treated as fixed, while only the growth rate (\(r\)) and midpoint year (\(t_0\)) are estimated. Parameter estimation is performed using nonlinear least squares optimisation through `curve_fit()` from SciPy. Initial parameter values are provided to improve convergence, while upper and lower bounds restrict estimates to plausible ranges.

If model fitting succeeds, the estimated parameters are used to generate predictions for all forecast years. If optimisation fails because of sparse or noisy data, the code falls back to stable default values for \(r\) and \(t_0\). This fallback makes sure that forecasting can continue rather than terminating with an error.

Finally, predicted values are constrained to remain between zero and the carrying capacity, rounded to integers, and returned together with the fitted growth rate, midpoint year, carrying capacity, and fitting status. The trained output model provides a bounded forecasting model in which EV keepership and charger counts increase realistically toward a long-run saturation level rather than growing indefinitely.


In [ ]:
from scipy.optimize import curve_fit

def logistic_curve(t, K, r, t0):
    return K / (1.0 + np.exp(-r * (t - t0)))


def fit_bounded_logistic(years, values, K, forecast_years):
    years = np.asarray(years, dtype=float)
    values = np.asarray(values, dtype=float)
    K = float(K)

    if len(years) < 3:
        raise ValueError("At least three observations are needed for bounded growth fitting.")

    # Ensure K is above observed values.
    if K <= values.max():
        K = values.max() * 1.2

    # Keep values inside the open interval (0, K) for curve fitting.
    values_fit = np.clip(values, 1, K - 1)

    def fixed_K_logistic(t, r, t0):
        return logistic_curve(t, K, r, t0)

    try:
        popt, _ = curve_fit(
            fixed_K_logistic,
            years,
            values_fit,
            p0=[0.35, np.median(years) + 5],
            bounds=([0.001, 2000], [3.0, 2080]),
            maxfev=50000
        )
        r, t0 = popt
        status = "fitted"
    except Exception as e:
        # Stable fallback if nonlinear fitting fails.
        r = 0.25
        t0 = np.median(years) + 5
        status = f"fallback: {str(e)[:80]}"

    all_years = np.asarray(forecast_years, dtype=float)
    preds = fixed_K_logistic(all_years, r, t0)
    preds = np.clip(preds, 0, K)
    preds = np.rint(preds).astype(int)

    return preds, float(r), float(t0), int(round(K)), status

# Forecast EV keepership and EV chargers to 2045

This cell generates **bounded logistic forecasts** for EV keepership and EV charger growth for every Welsh local authority district (LAD) under different capacity scenarios and projects values annually from **2026 to 2045**. The forecasting process uses the bounded logistic model defined previously and applies scenario-specific carrying capacities ((K)) stored in `capacity_df`.

First, the code defines the forecast horizon from **2026 to 2045**, producing yearly predictions immediately following the observed historical period. The forecasting process then iterates through each scenario contained in the capacity table (for example, `low`, `central`, and `high`). For each scenario, the corresponding LAD-specific carrying capacities are selected from `capacity_df`, and the observed EV keepership and charger time series for each LAD are retrieved from `panel_df` in chronological order.

For each LAD, two separate bounded logistic models are estimated. The first forecasts **EV keepership**, using observed EV keepership values together with the LAD-specific carrying capacity (`K_ev_keepership`). The second forecasts **EV charger growth**, using observed charger counts and the associated charger carrying capacity (`K_ev_chargers`). The function `fit_bounded_logistic()` is applied independently to both variables to estimate the logistic growth rate ((r)), midpoint year ((t_0)), and bounded yearly forecasts. If model fitting becomes unstable or fails because of sparse or noisy data, fallback parameter values are used to ensure that forecasting can continue.

The estimated model parameters are stored in a separate parameter table for each LAD, scenario, and target variable. These parameters include the scenario (`low`, `central`, `high`), LAD identifier and name, forecast target (`ev_keepership` or `ev_chargers`), carrying capacity ((K)), fitted growth rate ((r)), midpoint year ((t_0)), and model fitting status (`fitted` or fallback information).

At the same time, yearly forecasts are generated for every LAD and scenario between **2026 and 2045**, including predicted EV keepership and charger counts together with their associated carrying capacities. The code also calculates two additional indicators representing progress toward saturation: the **EV adoption ratio to carrying capacity**, calculated as EV keepership divided by EV keepership capacity, and the **charger ratio to carrying capacity**, calculated as charger count divided by charger capacity. These ratios indicate the proportion of the assumed carrying capacity reached in each year. For example, a value of (0.80) indicates that approximately **80%** of the assumed long-run saturation level has been achieved.

Finally, the resulting forecast table and model parameter table are exported as CSV files. The file `bounded_growth_forecasts_to_2053.csv` contains yearly EV and charger forecasts for all LADs and scenarios, while `bounded_growth_model_parameters.csv` stores the estimated logistic model parameters. Preview tables are displayed to support validation and interpretation of the generated forecasts and fitted model behaviour.


In [ ]:
forecast_years = list(range(2026, 2045))

bounded_rows = []
bounded_params = []

for scenario in sorted(capacity_df["scenario"].unique()):
    cap_s = capacity_df[capacity_df["scenario"] == scenario].copy()

    for _, cap in cap_s.iterrows():
        lad_uri = cap["lad_uri"]
        lad_name = cap["lad_name"]

        part = panel_df[panel_df["lad_uri"] == lad_uri].sort_values("year")

        if part.empty:
            continue

        years_obs = part["year"].values

        ev_preds, r_ev, t0_ev, K_ev, status_ev = fit_bounded_logistic(
            years=years_obs,
            values=part["ev_keepership"].values,
            K=int(cap["K_ev_keepership"]),
            forecast_years=forecast_years
        )

        charger_preds, r_ch, t0_ch, K_ch, status_ch = fit_bounded_logistic(
            years=years_obs,
            values=part["ev_chargers"].values,
            K=int(cap["K_ev_chargers"]),
            forecast_years=forecast_years
        )

        bounded_params.append({
            "scenario": scenario,
            "lad_uri": lad_uri,
            "lad_name": lad_name,
            "target": "ev_keepership",
            "K": K_ev,
            "r": r_ev,
            "t0": t0_ev,
            "fit_status": status_ev
        })

        bounded_params.append({
            "scenario": scenario,
            "lad_uri": lad_uri,
            "lad_name": lad_name,
            "target": "ev_chargers",
            "K": K_ch,
            "r": r_ch,
            "t0": t0_ch,
            "fit_status": status_ch
        })

        for year, ev_value, charger_value in zip(forecast_years, ev_preds, charger_preds):
            bounded_rows.append({
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "year": int(year),
                "ev_keepership": int(ev_value),
                "ev_chargers": int(charger_value),
                "K_ev_keepership": K_ev,
                "K_ev_chargers": K_ch
            })

bounded_forecast_df = pd.DataFrame(bounded_rows)
bounded_params_df = pd.DataFrame(bounded_params)

bounded_forecast_df["ev_adoption_ratio_to_K"] = (
    bounded_forecast_df["ev_keepership"] /
    bounded_forecast_df["K_ev_keepership"].replace(0, np.nan)
)

bounded_forecast_df["charger_ratio_to_K"] = (
    bounded_forecast_df["ev_chargers"] /
    bounded_forecast_df["K_ev_chargers"].replace(0, np.nan)
)

bounded_forecast_df.to_csv("bounded_growth_forecasts_to_2053.csv", index=False)
bounded_params_df.to_csv("bounded_growth_model_parameters.csv", index=False)

print("Forecast table:")
display(bounded_forecast_df.head(30))

print("Model parameter table:")
display(bounded_params_df.head(30))

print("Saved:")
print("- bounded_growth_forecasts_to_204.csv")
print("- bounded_growth_model_parameters.csv")

In [ ]:
test_years = [2023, 2024, 2025]

metrics_rows = []

# ------------------------------------------------------------
# Helper metric function
# ------------------------------------------------------------

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan

    return mae, rmse, r2


# ------------------------------------------------------------
# Bounded logistic: metrics only
# ------------------------------------------------------------

for scenario in sorted(capacity_df["scenario"].unique()):

    cap_s = capacity_df[capacity_df["scenario"] == scenario].copy()

    for _, cap in cap_s.iterrows():

        lad_uri = cap["lad_uri"]
        lad_name = cap["lad_name"]

        part = panel_df[panel_df["lad_uri"] == lad_uri].sort_values("year").copy()

        if len(part) < 3:
            continue

        for target, K_col in [
            ("ev_keepership", "K_ev_keepership"),
            ("ev_chargers", "K_ev_chargers")
        ]:

            y_true_all = []
            y_pred_all = []

            for test_year in test_years:

                train = part[part["year"] < test_year]
                test = part[part["year"] == test_year]

                if len(train) < 3 or test.empty:
                    continue

                logistic_pred, r, t0, K_used, status = fit_bounded_logistic(
                    years=train["year"].values,
                    values=train[target].values,
                    K=int(cap[K_col]),
                    forecast_years=[test_year]
                )

                y_true_all.append(test[target].iloc[0])
                y_pred_all.append(logistic_pred[0])

            if len(y_true_all) == 0:
                continue

            mae, rmse, r2 = compute_metrics(y_true_all, y_pred_all)

            metrics_rows.append({
                "model": "bounded_logistic",
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "target": target,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "n_test_predictions": len(y_true_all)
            })


# ------------------------------------------------------------
# LAD-level metrics table
# ------------------------------------------------------------

bounded_logistic_metrics_df = pd.DataFrame(metrics_rows)

display(
    bounded_logistic_metrics_df.sort_values(
        ["target", "lad_name", "scenario"]
    )
)


# ------------------------------------------------------------
# Aggregate metrics summary
# ------------------------------------------------------------

bounded_logistic_summary_df = (
    bounded_logistic_metrics_df
    .groupby(["model", "scenario", "target"], as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean"),
        n_LADs=("lad_name", "nunique"),
        n_test_predictions=("n_test_predictions", "sum")
    )
)

display(
    bounded_logistic_summary_df.sort_values(
        ["target", "mean_RMSE"]
    )
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

bounded_logistic_metrics_df.to_csv(
    "bounded_logistic_lad_level_metrics.csv",
    index=False
)

bounded_logistic_summary_df.to_csv(
    "bounded_logistic_summary_metrics.csv",
    index=False
)

print("Saved:")
print("- bounded_logistic_lad_level_metrics.csv")
print("- bounded_logistic_summary_metrics.csv")

# Neural Network Forecast Construction

This cell constructs a **standalone neural network forecasting model** to predict future EV keepership and EV charger growth for each Welsh local authority district (LAD) from **2026 to 2053**. Unlike the bounded logistic model, which constrains growth using a carrying-capacity parameter (\(K\)), the neural network model is entirely **data-driven** and learns nonlinear temporal relationships directly from historical observations.

The neural network is implemented using **Multi-Layer Perceptron Regression (`MLPRegressor`)** from Scikit-learn. The objective is to learn a nonlinear mapping between time and EV-related outcomes:

\[
f(\text{year})
\rightarrow
\text{EV keepership or charger count}
\]

For each LAD, the model uses historical yearly observations extracted from `panel_df`, where:

\[X=\text{year}\] and \[y=\text{EV keepership or EV charger count}\]

Thus, the model attempts to learn how EV keepership and charger counts evolve through time without assuming a predefined growth structure.

## 1. Neural Network Architecture

The neural network is constructed using a Scikit-learn pipeline:

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(...))
])

In [ ]:
# Scikit-learn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics.pairwise import cosine_similarity

def fit_neural_network(years, values, forecast_years):
    years = np.asarray(years, dtype=float).reshape(-1, 1)
    values = np.asarray(values, dtype=float)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(years, values)

    forecast_years_arr = np.asarray(forecast_years, dtype=float).reshape(-1, 1)
    preds = model.predict(forecast_years_arr)

    preds = np.clip(preds, 0, None)
    preds = np.rint(preds).astype(int)

    return preds, model


# ------------------------------------------------------------
# Forecast years
# ------------------------------------------------------------

forecast_years = list(range(2026, 2045))


# ------------------------------------------------------------
# Fit one neural network per LAD and per target variable
# ------------------------------------------------------------

nn_forecast_rows = []

for (lad_uri, lad_name), part in panel_df.groupby(["lad_uri", "lad_name"]):

    part = part.sort_values("year").copy()

    # Skip LADs with too few observations
    if len(part) < 3:
        continue

    years_obs = part["year"].values

    # EV keepership forecast
    ev_preds, ev_model = fit_neural_network(
        years=years_obs,
        values=part["ev_keepership"].values,
        forecast_years=forecast_years
    )

    # EV charger forecast
    charger_preds, charger_model = fit_neural_network(
        years=years_obs,
        values=part["ev_chargers"].values,
        forecast_years=forecast_years
    )

    for year, ev_value, charger_value in zip(
        forecast_years,
        ev_preds,
        charger_preds
    ):
        nn_forecast_rows.append({
            "lad_uri": lad_uri,
            "lad_name": lad_name,
            "year": int(year),
            "ev_keepership": int(ev_value),
            "ev_chargers": int(charger_value),
            "model": "neural_network"
        })


# ------------------------------------------------------------
# Create forecast dataframe
# ------------------------------------------------------------

nn_forecast_df = pd.DataFrame(nn_forecast_rows)

display(nn_forecast_df.head(30))

nn_forecast_df.to_csv(
    "neural_network_forecasts_to_2045.csv",
    index=False
)

print("Neural network forecast shape:", nn_forecast_df.shape)
print("Saved: neural_network_forecasts_to_2045.csv")

## Neural network backtesting accuracy

This cell evaluates the predictive accuracy of the neural network model using a **historical backtesting procedure**. Instead of directly forecasting unknown future years, the model is tested on years for which observed values are already available (**2023–2025**) to assess how accurately it reproduces known EV trends.

The analysis is conducted separately for **EV keepership** and **EV charger counts** across Welsh local authority districts (LADs). For each LAD, historical observations stored in `panel_df` are sorted chronologically and divided into training and testing periods.

The neural network model is evaluated using a rolling forecasting strategy:

* **2023** is predicted using only observations before 2023.
* **2024** is predicted using only observations before 2024.
* **2025** is predicted using only observations before 2025.

This approach simulates a realistic forecasting setting in which the model only has access to historical information available prior to the prediction year.

For each test year, the neural network (`fit_neural_network`) is trained on historical observations and used to predict the held-out year. Predicted values are then compared with observed values to assess forecasting performance.

Three forecasting performance measures are calculated.

### Mean Absolute Error (MAE)

The **Mean Absolute Error (MAE)** measures the average magnitude of forecasting error:

$$
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
$$

where:

* $y_i$ = observed value
* $\hat{y}_i$ = predicted value
* $n$ = number of test observations

### Root Mean Squared Error (RMSE)

The **Root Mean Squared Error (RMSE)** gives greater weight to larger forecasting errors:

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
$$

### Coefficient of Determination ($R^2$)

The **coefficient of determination ($R^2$)** measures how well predictions explain variation in observed values:

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat{y}*i)^2}{\sum*{i=1}^{n}(y_i-\bar{y})^2}
$$

where:

* $\bar{y}$ = mean observed value

The resulting LAD-level performance metrics are stored in `nn_metrics_df`, while an overall performance summary is produced by averaging MAE, RMSE, and $R^2$ values across Welsh LADs separately for EV keepership and EV charger forecasting.

Finally, the detailed results are exported to `neural_network_accuracy_metrics.csv` for further analysis and reporting. In practical terms, this cell answers the following question:

> **How accurately would the neural network have predicted recent years if it had been used historically for EV forecasting?**


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

test_years = [2023, 2024, 2025]

metric_rows = []

# ------------------------------------------------------------
# Neural network: LAD-level metrics
# ------------------------------------------------------------

for (lad_uri, lad_name), part in panel_df.groupby(["lad_uri", "lad_name"]):

    part = part.sort_values("year")

    for target in ["ev_keepership", "ev_chargers"]:

        y_true_all = []
        y_pred_all = []

        for test_year in test_years:

            train = part[part["year"] < test_year]
            test = part[part["year"] == test_year]

            if len(train) < 3 or test.empty:
                continue

            pred, _ = fit_neural_network(
                years=train["year"].values,
                values=train[target].values,
                forecast_years=[test_year]
            )

            y_true_all.append(test[target].iloc[0])
            y_pred_all.append(pred[0])

        if len(y_true_all) == 0:
            continue

        mae = mean_absolute_error(y_true_all, y_pred_all)
        rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        r2 = (
            r2_score(y_true_all, y_pred_all)
            if len(y_true_all) > 1
            else np.nan
        )

        metric_rows.append({
            "model": "neural_network",
            "scenario": "none",
            "lad_uri": lad_uri,
            "lad_name": lad_name,
            "target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "n_test_predictions": len(y_true_all)
        })


# ------------------------------------------------------------
# LAD-level metrics table
# ------------------------------------------------------------

nn_metrics_df = pd.DataFrame(metric_rows)

display(
    nn_metrics_df.sort_values(
        ["target", "lad_name"]
    )
)


# ------------------------------------------------------------
# Aggregate summary (same format as bounded logistic)
# ------------------------------------------------------------

nn_summary_df = (
    nn_metrics_df
    .groupby(["model", "scenario", "target"], as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean"),
        n_LADs=("lad_name", "nunique"),
        n_test_predictions=("n_test_predictions", "sum")
    )
)

nn_summary_df = (
    nn_summary_df
    .sort_values(["target", "mean_RMSE"])
    .reset_index()
)

display(nn_summary_df)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

nn_metrics_df.to_csv(
    "neural_network_lad_level_metrics.csv",
    index=False
)

nn_summary_df.to_csv(
    "neural_network_summary_metrics.csv",
    index=False
)

print("Saved:")
print("- neural_network_lad_level_metrics.csv")
print("- neural_network_summary_metrics.csv")

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def fit_scenario_aware_nn(years, values, capacities, forecast_years, forecast_capacities):
    """
    Fits a neural network using both temporal (years) and scenario (capacities) features.
    """
    # Prepare training features: [year, K]
    X_train = np.column_stack([years, capacities]).astype(float)
    y_train = np.asarray(values, dtype=float)

    # Prepare forecast features: [year, K_forecast]
    X_forecast = np.column_stack([forecast_years, forecast_capacities]).astype(float)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    preds = model.predict(X_forecast)
    preds = np.clip(preds, 0, None)
    return np.rint(preds).astype(int)

# --- Example execution for all LADs and scenarios ---
scenario_nn_forecasts = []
forecast_years = list(range(2026, 2046))

for scenario in capacity_df['scenario'].unique():
    cap_subset = capacity_df[capacity_df['scenario'] == scenario]

    for target in ["ev_keepership", "ev_chargers"]:
        K_col = "K_ev_keepership" if target == "ev_keepership" else "K_ev_chargers"

        for _, cap_row in cap_subset.iterrows():
            lad_uri = cap_row['lad_uri']
            lad_name = cap_row['lad_name']
            K_val = cap_row[K_col]

            # Get historical data
            hist = panel_df[panel_df['lad_uri'] == lad_uri].sort_values('year')
            if len(hist) < 3: continue

            # For training, historical capacity is assumed to be the current K (scenario-aware learning)
            train_caps = np.full(len(hist), K_val)
            forecast_caps = np.full(len(forecast_years), K_val)

            preds = fit_scenario_aware_nn(
                years=hist['year'].values,
                values=hist[target].values,
                capacities=train_caps,
                forecast_years=forecast_years,
                forecast_capacities=forecast_caps
            )

            for yr, p in zip(forecast_years, preds):
                scenario_nn_forecasts.append({
                    "model": "scenario_aware_nn",
                    "scenario": scenario,
                    "lad_name": lad_name,
                    "year": yr,
                    "target": target,
                    "prediction": p
                })

scenario_nn_df = pd.DataFrame(scenario_nn_forecasts)
display(scenario_nn_df.head())
print("Scenario-aware Neural Network forecasts generated.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

test_years = [2023, 2024, 2025]
scenarios = capacity_df["scenario"].unique()

backtest_results = []

# ------------------------------------------------------------
# Walk-forward backtesting
# ------------------------------------------------------------

for scenario in scenarios:

    cap_subset = capacity_df[
        capacity_df["scenario"] == scenario
    ]

    for target in ["ev_keepership", "ev_chargers"]:

        K_col = (
            "K_ev_keepership"
            if target == "ev_keepership"
            else "K_ev_chargers"
        )

        for test_year in test_years:

            for _, cap_row in cap_subset.iterrows():

                lad_uri = cap_row["lad_uri"]
                lad_name = cap_row["lad_name"]
                K_val = cap_row[K_col]

                # historical LAD series
                full_series = (
                    panel_df[
                        panel_df["lad_uri"] == lad_uri
                    ]
                    .sort_values("year")
                )

                train = full_series[
                    full_series["year"] < test_year
                ]

                test = full_series[
                    full_series["year"] == test_year
                ]

                if len(train) < 3 or test.empty:
                    continue

                train_caps = np.full(
                    len(train),
                    K_val
                )

                test_cap = np.array([K_val])

                pred = fit_scenario_aware_nn(
                    years=train["year"].values,
                    values=train[target].values,
                    capacities=train_caps,
                    forecast_years=[test_year],
                    forecast_capacities=test_cap
                )

                y_true = float(test[target].iloc[0])
                y_pred = float(np.asarray(pred).ravel()[0])

                backtest_results.append({
                    "model": "scenario_aware_nn",
                    "scenario": scenario,
                    "target": target,
                    "test_year": test_year,
                    "lad_uri": lad_uri,
                    "lad_name": lad_name,
                    "actual": y_true,
                    "predicted": y_pred
                })


backtest_df = pd.DataFrame(backtest_results)


# ------------------------------------------------------------
# LAD-level metrics
# ------------------------------------------------------------

metric_rows = []

for (scenario, target, lad_uri, lad_name), group in (
    backtest_df.groupby(
        ["scenario", "target", "lad_uri", "lad_name"]
    )
):

    y_true = group["actual"].astype(float)
    y_pred = group["predicted"].astype(float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = (
        r2_score(y_true, y_pred)
        if len(y_true) > 1
        else np.nan
    )

    accuracy_percent = (
        max(0, r2 * 100)
        if not np.isnan(r2)
        else np.nan
    )

    metric_rows.append({
        "model": "scenario_aware_nn",
        "scenario": scenario,
        "lad_uri": lad_uri,
        "lad_name": lad_name,
        "target": target,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "accuracy_percent": accuracy_percent,
        "n_test_predictions": len(group)
    })


scenario_nn_metrics_df = pd.DataFrame(metric_rows)

display(
    scenario_nn_metrics_df.sort_values(
        ["target", "lad_name", "scenario"]
    )
)


# ------------------------------------------------------------
# Summary accuracy table
# Same format as other models
# ------------------------------------------------------------

scenario_nn_accuracy_summary_df = (
    scenario_nn_metrics_df
    .groupby(
        ["model", "scenario", "target"],
        as_index=False
    )
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean"),
        accuracy_percent=(
            "accuracy_percent",
            "mean"
        ),
        n_LADs=("lad_name", "nunique"),
        n_test_predictions=(
            "n_test_predictions",
            "sum"
        )
    )
)

scenario_nn_accuracy_summary_df = (
    scenario_nn_accuracy_summary_df
    .sort_values(
        ["target", "mean_RMSE"]
    )
    .reset_index()
)

display(scenario_nn_accuracy_summary_df)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

scenario_nn_metrics_df.to_csv(
    "scenario_aware_nn_lad_level_metrics.csv",
    index=False
)

scenario_nn_accuracy_summary_df.to_csv(
    "scenario_aware_nn_summary_metrics.csv",
    index=False
)

print("\nBacktesting complete for Scenario-Aware Neural Network.")
print("Saved:")
print("- scenario_aware_nn_lad_level_metrics.csv")
print("- scenario_aware_nn_summary_metrics.csv")

# Generate EV keepership predictions for 2045 using scenario aware neural network, scenario = central  because it is the best performing model

In [ ]:
# ------------------------------------------------------------
# Generate EV keepership predictions to 2045 only
# ------------------------------------------------------------

forecast_years = list(range(2026, 2046))
target = "ev_keepership"
K_col = "K_ev_keepership"

forecast_rows = []

for scenario in scenarios:

    cap_subset = capacity_df[
        capacity_df["scenario"] == scenario
    ]

    for _, cap_row in cap_subset.iterrows():

        lad_uri = cap_row["lad_uri"]
        lad_name = cap_row["lad_name"]
        K_val = cap_row[K_col]

        full_series = (
            panel_df[
                panel_df["lad_uri"] == lad_uri
            ]
            .sort_values("year")
        )

        train = full_series[
            full_series["year"] < 2026
        ]

        if len(train) < 3:
            continue

        forecast_capacities = np.full(
            len(forecast_years),
            K_val
        )

        pred = fit_scenario_aware_nn(
            years=train["year"].values,
            values=train[target].values,
            capacities=np.full(len(train), K_val),
            forecast_years=forecast_years,
            forecast_capacities=forecast_capacities
        )

        pred = np.maximum(pred, 0)
        pred = np.rint(pred)

        for year, y_pred in zip(forecast_years, pred):

            forecast_rows.append({
                "model": "scenario_aware_nn",
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "year": year,
                "target": target,
                "predicted_ev_keepership": float(y_pred)
            })


ev_keepership_predictions_2045_df = pd.DataFrame(forecast_rows)

display(
    ev_keepership_predictions_2045_df.sort_values(
        ["scenario", "lad_name", "year"]
    )
)

ev_keepership_predictions_2045_df.to_csv(
    "ev_keepership_predictions_to_2045.csv",
    index=False
)

print("Saved: ev_keepership_predictions_to_2045.csv")

In [ ]:
files.download('ev_keepership_predictions_to_2045.csv')

## Graph Neural Network (GNN) backtesting accuracy

This cell evaluates the predictive accuracy of a **Graph Neural Network (GNN)** using a **historical backtesting framework**. Instead of directly forecasting unknown future years, the model is tested on years for which observed values are already available (**2023–2025**) to assess how accurately it reproduces known electric vehicle (EV) trends.

The analysis is conducted separately for **EV keepership** and **EV charger counts** across Welsh local authority districts (LADs). Unlike the standard neural network and bounded logistic models, which treat each LAD independently, the GNN learns EV growth patterns **across Welsh LADs simultaneously** by allowing information to be shared between LADs exhibiting similar historical EV trajectories.

The model is evaluated using a rolling forecasting strategy:

* **2023** is predicted using only observations before 2023;
* **2024** is predicted using only observations before 2024; and
* **2025** is predicted using only observations before 2025.

This approach simulates a realistic forecasting setting in which the model only has access to historical information available prior to the prediction year, thereby avoiding information leakage from future observations.

The model first constructs a lookup table of Welsh LADs and creates a graph in which each LAD is represented as a **node**. Historical EV observations are then converted into a temporal feature matrix. For each LAD, yearly EV keepership and charger observations from the training period are concatenated into a feature vector:

$$
X_i
===

\left[
\mathrm{EV\ keepership}*{2019},
\mathrm{EV\ chargers}*{2019},
\ldots,
\mathrm{EV\ keepership}*{t},
\mathrm{EV\ chargers}*{t}
\right]
$$

where:

* $X_i$ represents the historical EV feature vector for LAD $i$; and
* $t$ represents the most recent historical year used for training.

To stabilise large numerical differences and reduce skewness in EV counts, feature values are transformed using the logarithmic function `log1p()` before model fitting. Missing yearly observations are represented as zeros within the feature matrix.

A similarity graph is subsequently constructed using **pairwise cosine similarity** between LAD feature vectors:

$$
\mathrm{Similarity}(i,j)
========================

\frac{
X_i \cdot X_j
}{
\left| X_i \right|
\left| X_j \right|
}
$$

where larger values indicate more comparable historical EV growth trajectories between LADs.

To reduce over-connectivity and preserve only meaningful relationships, each LAD is connected only to its **five most similar neighbours** (`top_k = 5`). Negative similarities are removed, and **self-loops** are added so that each LAD also retains access to its own historical information during model learning.

The adjacency matrix is subsequently normalised according to:

$$
\hat{A}
=======

D^{-1/2} A D^{-1/2}
$$

where:

* $A$ represents the adjacency matrix; and
* $D$ represents the degree matrix.

This normalisation stabilises information propagation and prevents highly connected LADs from disproportionately influencing model learning.

The forecasting architecture is implemented as a **two-layer Graph Convolutional Network (GCN)** using custom `GraphConvolution` layers. During each forward pass, information is propagated across the graph according to:

$$
H
=

\hat{A}X
$$

thereby enabling LAD representations to be updated using information from historically similar local authorities. Two graph-convolution layers with **ReLU activation functions** are used to capture nonlinear relationships between historical EV trajectories and future EV outcomes, followed by a dense output layer for prediction.

Model optimisation is performed using **backpropagation** with the **Adam optimiser**, while **Mean Squared Error (MSE)** is used as the optimisation objective. Model parameters are iteratively updated over **1000 epochs** to minimise forecasting error between predicted and observed EV outcomes.

For each backtesting year and target variable, the trained GNN produces predictions for all LADs simultaneously. Predictions are inverse-transformed using `expm1()` to return forecasts to the original count scale, constrained to non-negative values, and rounded to integer counts before comparison with observed values.

Three forecasting performance measures are calculated.

### Mean Absolute Error (MAE)

The **Mean Absolute Error (MAE)** measures the average magnitude of forecasting error:

$$
\mathrm{MAE}
============

\frac{1}{n}
\sum_{i=1}^{n}
\left|
y_i-\hat{y}_i
\right|
$$

where:

* $y_i$ = observed value
* $\hat{y}_i$ = predicted value
* $n$ = number of test observations

### Root Mean Squared Error (RMSE)

The **Root Mean Squared Error (RMSE)** gives greater weight to larger forecasting errors:

$$
\mathrm{RMSE}
=============

\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
y_i-\hat{y}_i
\right)^2
}
$$

### Coefficient of Determination ($R^2$)

The **coefficient of determination ($R^2$)** measures how well predictions explain variation in observed values:

$$
R^2
===

## 1

\frac{
\sum_{i=1}^{n}
\left(
y_i-\hat{y}*i
\right)^2
}{
\sum*{i=1}^{n}
\left(
y_i-\bar{y}
\right)^2
}
$$

where:

* $\bar{y}$ = mean observed value

LAD-level forecasting performance metrics are stored in `gnn_metrics_df`, while an overall performance summary is produced by averaging MAE, RMSE, and $R^2$ values across Welsh LADs separately for EV keepership and EV charger forecasting.

Finally, detailed results are exported to `graph_neural_network_lad_level_accuracy_metrics.csv`, while summary metrics are exported to `graph_neural_network_summary_accuracy_metrics.csv` for further analysis and reporting. In practical terms, this cell answers the following question:

> **How accurately would a Graph Neural Network have predicted recent EV trends across Welsh LADs if it had been used historically for forecasting?**


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

test_years = [2023, 2024, 2025]
targets = ["ev_keepership", "ev_chargers"]
top_k = 5
epochs = 1000
learning_rate = 0.01

torch.manual_seed(42)
np.random.seed(42)


# ------------------------------------------------------------
# 1. LAD lookup
# ------------------------------------------------------------

lad_lookup = (
    panel_df[["lad_uri", "lad_name"]]
    .drop_duplicates()
    .sort_values("lad_uri")
    .reset_index(drop=True)
)

lad_uris = lad_lookup["lad_uri"].tolist()
lad_to_idx = {lad: i for i, lad in enumerate(lad_uris)}
n_lads = len(lad_uris)


# ------------------------------------------------------------
# 2. Build temporal feature matrix: keepership + chargers by year
# ------------------------------------------------------------

def build_temporal_feature_matrix(panel_df, train_years):
    rows = []

    for lad_uri in lad_uris:

        part = (
            panel_df[panel_df["lad_uri"] == lad_uri]
            .sort_values("year")
        )

        values_by_year = {
            year: row
            for year, row in part.set_index("year").iterrows()
        }

        features = []

        for year in train_years:

            if year in values_by_year:
                keepership = values_by_year[year]["ev_keepership"]
                chargers = values_by_year[year]["ev_chargers"]
            else:
                keepership = 0
                chargers = 0

            features.extend([
                np.log1p(float(keepership)),
                np.log1p(float(chargers))
            ])

        rows.append(features)

    return np.asarray(rows, dtype=np.float32)


# ------------------------------------------------------------
# 3. Build similarity graph from LAD histories
# ------------------------------------------------------------

def build_similarity_adjacency(X, top_k=5):
    similarity = cosine_similarity(X)

    A = np.zeros_like(similarity, dtype=np.float32)

    for i in range(similarity.shape[0]):

        neighbours = np.argsort(similarity[i])[::-1]

        neighbours = [
            j for j in neighbours
            if j != i and similarity[i, j] > 0
        ][:top_k]

        for j in neighbours:
            A[i, j] = similarity[i, j]
            A[j, i] = similarity[i, j]

    # self-loops
    np.fill_diagonal(A, 1.0)

    degree = np.sum(A, axis=1)
    degree[degree == 0] = 1.0

    D_inv_sqrt = np.diag(1.0 / np.sqrt(degree))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return A_norm.astype(np.float32)


# ------------------------------------------------------------
# 4. Build labels for one target and one test year
# ------------------------------------------------------------

def build_labels(panel_df, target, test_year):
    y = np.full(n_lads, np.nan, dtype=np.float32)

    for lad_uri in lad_uris:

        part = panel_df[
            (panel_df["lad_uri"] == lad_uri) &
            (panel_df["year"] == test_year)
        ]

        if not part.empty:
            y[lad_to_idx[lad_uri]] = np.log1p(
                float(part[target].iloc[0])
            )

    return y


# ------------------------------------------------------------
# 5. Simple two-layer GCN
# ------------------------------------------------------------

class GraphConvolution(nn.Module):
    """Dense graph convolution: A_norm @ X @ W (+ activation). PyTorch port of the
    earlier Keras layer; PyTorch is what the setup cell installs, so no TensorFlow is needed."""

    def __init__(self, in_dim, units, activation=None):
        super().__init__()
        self.W = nn.Parameter(torch.empty(in_dim, units))
        nn.init.xavier_uniform_(self.W)          # glorot_uniform, as before
        self.activation = activation

    def forward(self, X, A):
        HW = A @ X @ self.W
        return self.activation(HW) if self.activation is not None else HW


class GCNForecaster(nn.Module):
    """Two-layer GCN (32, 16 units, ReLU) with a linear output, as in the original model."""

    def __init__(self, in_dim):
        super().__init__()
        self.gc1 = GraphConvolution(in_dim, 32, torch.relu)
        self.gc2 = GraphConvolution(32, 16, torch.relu)
        self.out = nn.Linear(16, 1)
        nn.init.xavier_uniform_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, X, A):
        return self.out(self.gc2(self.gc1(X, A), A))


def fit_gcn_forecaster(X, A_norm, y, epochs=1000, learning_rate=0.01):
    """Fit the GCN on log1p targets (masked MSE over the LADs with a label) and return
    rounded count predictions for every LAD."""
    valid_mask = ~np.isnan(y)

    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X).astype(np.float32)

    X_t = torch.tensor(X_scaled, dtype=torch.float32)
    A_t = torch.tensor(np.asarray(A_norm, dtype=np.float32))
    y_t = torch.tensor(np.nan_to_num(y, nan=0.0).reshape(-1, 1), dtype=torch.float32)
    mask_t = torch.tensor(valid_mask.reshape(-1, 1), dtype=torch.bool)

    torch.manual_seed(SEED)
    model = GCNForecaster(X.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_t, A_t)
        loss = torch.mean((y_t[mask_t] - pred[mask_t]) ** 2)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_log = model(X_t, A_t).numpy().ravel()

    # inverse log1p transform and round to counts
    pred = np.expm1(pred_log)
    pred = np.maximum(pred, 0)
    pred = np.rint(pred)

    return pred


# ------------------------------------------------------------
# 6. Walk-forward backtesting: 2023--2025
# ------------------------------------------------------------

prediction_store = {
    (lad_uri, target): {"y_true": [], "y_pred": []}
    for lad_uri in lad_uris
    for target in targets
}

for test_year in test_years:

    train_years = sorted(
        panel_df.loc[
            panel_df["year"] < test_year,
            "year"
        ].unique()
    )

    if len(train_years) < 3:
        continue

    X = build_temporal_feature_matrix(
        panel_df=panel_df,
        train_years=train_years
    )

    A_norm = build_similarity_adjacency(
        X=X,
        top_k=top_k
    )

    for target in targets:

        y_log = build_labels(
            panel_df=panel_df,
            target=target,
            test_year=test_year
        )

        pred_all = fit_gcn_forecaster(
            X=X,
            A_norm=A_norm,
            y=y_log,
            epochs=epochs,
            learning_rate=learning_rate
        )

        for lad_uri in lad_uris:

            idx = lad_to_idx[lad_uri]

            true_row = panel_df[
                (panel_df["lad_uri"] == lad_uri) &
                (panel_df["year"] == test_year)
            ]

            if true_row.empty or np.isnan(y_log[idx]):
                continue

            y_true = float(true_row[target].iloc[0])
            y_pred = float(np.asarray(pred_all[idx]).ravel()[0])

            prediction_store[(lad_uri, target)]["y_true"].append(y_true)
            prediction_store[(lad_uri, target)]["y_pred"].append(y_pred)


# ------------------------------------------------------------
# 7. LAD-level metrics
# ------------------------------------------------------------

metric_rows = []

for lad_uri in lad_uris:

    lad_name = lad_lookup.loc[
        lad_lookup["lad_uri"] == lad_uri,
        "lad_name"
    ].iloc[0]

    for target in targets:

        y_true_all = np.asarray(
            prediction_store[(lad_uri, target)]["y_true"],
            dtype=float
        )

        y_pred_all = np.asarray(
            prediction_store[(lad_uri, target)]["y_pred"],
            dtype=float
        )

        if len(y_true_all) == 0:
            continue

        mae = mean_absolute_error(y_true_all, y_pred_all)
        rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        r2 = r2_score(y_true_all, y_pred_all) if len(y_true_all) > 1 else np.nan

        accuracy_percent = (
            max(0, r2 * 100)
            if not np.isnan(r2)
            else np.nan
        )

        metric_rows.append({
            "model": "graph_neural_network",
            "scenario": "none",
            "lad_uri": lad_uri,
            "lad_name": lad_name,
            "target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "accuracy_percent": accuracy_percent,
            "n_test_predictions": len(y_true_all)
        })


gnn_metrics_df = pd.DataFrame(metric_rows)

display(
    gnn_metrics_df.sort_values(
        ["target", "lad_name"]
    )
)


# ------------------------------------------------------------
# 8. Summary table — same format as other models
# ------------------------------------------------------------

gnn_accuracy_summary_df = (
    gnn_metrics_df
    .groupby(["model", "scenario", "target"], as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean"),
        accuracy_percent=("accuracy_percent", "mean"),
        n_LADs=("lad_name", "nunique"),
        n_test_predictions=("n_test_predictions", "sum")
    )
)

gnn_accuracy_summary_df = (
    gnn_accuracy_summary_df
    .sort_values(["target", "mean_RMSE"])
    .reset_index()
)

display(gnn_accuracy_summary_df)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

gnn_metrics_df.to_csv(
    "graph_neural_network_lad_level_accuracy_metrics.csv",
    index=False
)

gnn_accuracy_summary_df.to_csv(
    "graph_neural_network_summary_accuracy_metrics.csv",
    index=False
)

print("Saved:")
print("- graph_neural_network_lad_level_accuracy_metrics.csv")
print("- graph_neural_network_summary_accuracy_metrics.csv")


# Scenario-aware Graph Neural Network (GNN)

To evaluate whether externally defined growth assumptions and relational learning between Welsh local authority districts (LADs) could improve EV forecasting performance, a **scenario-aware Graph Neural Network (GNN)** was implemented in addition to the bounded logistic model, standard neural network, and scenario-aware neural network. The purpose of this model was to condition graph-based forecasts on scenario-specific carrying-capacity values while simultaneously enabling the model to learn from similarities shared across LADs. This allowed forecasting performance to be evaluated under alternative assumptions regarding future EV keepership and EV charger growth while incorporating interdependencies between historically similar Welsh local authorities.

The model was evaluated under three capacity scenarios: **low, central, and high**. For each scenario, LAD-specific carrying-capacity values were obtained from the capacity table. Separate capacity parameters were used for the two forecasting targets. For EV keepership, the model used (K_{\mathrm{EV}}), stored as \texttt{K_ev_keepership}; for EV chargers, it used (K_{\mathrm{CH}}), stored as \texttt{K_ev_chargers}. Thus, for each LAD (i), scenario (s), and target variable (y), the model was conditioned on a fixed scenario-specific carrying capacity:

[
K_{i,s}^{(y)}
=============

\begin{cases}
K_{i,s}^{\mathrm{EV}}, & \text{if } y = \text{EV keepership}, \
K_{i,s}^{\mathrm{CH}}, & \text{if } y = \text{EV chargers}.
\end{cases}
]

For each LAD, the historical time series was ordered by year and split using a walk-forward backtesting procedure. For a given test year (T \in {2023,2024,2025}), all observations before (T) were used for training:

[
\mathcal{D}_{\mathrm{train}}
============================

{(t,y_{i,t},K_{i,s}^{(y)}):t<T},
]

while the observation in year (T) was held out for testing:

[
\mathcal{D}_{\mathrm{test}}
===========================

{(T,y_{i,T},K_{i,s}^{(y)})}.
]

Only LADs with at least three historical training observations were included in each fold. This ensured that each prediction was based on a minimum temporal history and that no future observations were used during model fitting.

Unlike the scenario-aware neural network, which learns independently for each LAD, the scenario-aware GNN represents Welsh LADs as an interconnected graph in which each LAD forms a node and edges represent historical similarity between LADs. Historical EV observations were transformed into a temporal feature matrix in which yearly EV keepership and charger observations were concatenated into a feature vector:

[
X_i
===

\left[
\mathrm{EV\ keepership}*{2019},
\mathrm{EV\ chargers}*{2019},
\ldots,
\mathrm{EV\ keepership}*{t},
\mathrm{EV\ chargers}*{t}
\right].
]

To incorporate externally defined growth assumptions, scenario-specific carrying-capacity values were appended to the LAD feature representation:

[
X_{i,s}^{*}
===========

\left[
X_i,
K_{i,s}^{\mathrm{EV}},
K_{i,s}^{\mathrm{CH}}
\right].
]

Thus, the scenario-aware GNN jointly learned from historical EV trajectories and scenario-specific carrying-capacity assumptions.

To stabilise large count differences and reduce skewness in the observed EV data, feature values were transformed using \texttt{log1p()} prior to model fitting.

A similarity graph was subsequently constructed using pairwise cosine similarity between LAD feature vectors:

[
\mathrm{Similarity}(i,j)
========================

\frac{
X_{i,s}^{*}
\cdot
X_{j,s}^{*}
}{
\left|X_{i,s}^{*}\right|
\left|X_{j,s}^{*}\right|
}.
]

where larger values indicate more comparable historical EV growth trajectories and scenario-adjusted characteristics between LADs. To reduce over-connectivity and preserve only meaningful relationships, each LAD was connected to its five most similar neighbours (\texttt{top_k = 5}). Negative similarities were removed and self-loops were added so that each LAD retained access to its own historical information.

The adjacency matrix was subsequently normalised according to:

[
\hat{A}
=======

D^{-1/2}
A
D^{-1/2},
]

where (A) represents the adjacency matrix and (D) the degree matrix. This normalisation stabilises information propagation and prevents highly connected LADs from disproportionately influencing model learning.

The forecasting architecture was implemented as a **two-layer Graph Convolutional Network (GCN)** using custom \texttt{GraphConvolution} layers. During each forward pass, information was propagated across the graph according to:

[
H
=

\hat{A}X,
]

thereby enabling LAD representations to be updated using information from historically similar local authorities. Hidden layers using Rectified Linear Unit (ReLU) activation functions were employed to capture nonlinear relationships between historical EV trajectories, scenario-specific carrying-capacity assumptions, and future EV outcomes.

Conceptually, the scenario-aware GNN learned a mapping of the form:

[
\hat{y}_{i,T}
=============

f_{\theta}
\left(
X_{i,s}^{*},
\hat{A}
\right),
]

where (X_{i,s}^{*}) denotes the scenario-adjusted LAD feature matrix, (\hat{A}) represents the similarity graph, and (\theta) denotes the trainable neural network parameters. This enabled forecasts to be conditioned on both temporal EV dynamics and graph-based relational structure under alternative growth assumptions.

Model optimisation was performed using **backpropagation** with the **Adam optimiser** and **Mean Squared Error (MSE)** loss over **1000 epochs**. Following training, predictions were inverse-transformed using \texttt{expm1()} to return forecasts to the original count scale, constrained to non-negative values, rounded to integers, and evaluated using **Mean Absolute Error (MAE)**, **Root Mean Squared Error (RMSE)**, and the **coefficient of determination ((R^2))** within a walk-forward backtesting framework covering **2023--2025**.

Forecasting performance was first calculated at LAD level. For each LAD, scenario, and target, predictions from the 2023--2025 walk-forward evaluation were grouped and compared with observed values using:

[
\mathrm{MAE}
============

\frac{1}{n}
\sum_{j=1}^{n}
\left|
y_j-\hat{y}_j
\right|,
]

[
\mathrm{RMSE}
=============

\sqrt{
\frac{1}{n}
\sum_{j=1}^{n}
\left(
y_j-\hat{y}_j
\right)^2
},
]

[
R^2
===

## 1

\frac{
\sum_{j=1}^{n}
(y_j-\hat{y}*j)^2
}{
\sum*{j=1}^{n}
(y_j-\bar{y})^2
}.
]

An (R^2)-derived accuracy percentage was also computed for consistency with the other model comparison tables:

[
\mathrm{Accuracy}(%)
====================

\max(0,R^2\times100).
]

Negative (R^2) values therefore indicate that the model performed worse than a naïve mean predictor and were truncated to zero when expressed as an accuracy percentage.

Finally, LAD-level results were aggregated by model, scenario, and target variable. The summary table reports mean MAE, mean RMSE, mean (R^2), mean (R^2)-derived accuracy percentage, the number of Welsh LADs evaluated, and the total number of test predictions. This enabled direct comparison between the scenario-aware GNN, bounded logistic model, standard neural network, graph neural network, and scenario-aware neural network.


Scenario-Aware Graph Neural Network Forecasting

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

test_years = [2023, 2024, 2025]
targets = ["ev_keepership", "ev_chargers"]
scenarios = capacity_df["scenario"].unique()

top_k = 5
epochs = 1000
learning_rate = 0.01

torch.manual_seed(42)
np.random.seed(42)


# ------------------------------------------------------------
# 1. LAD lookup
# ------------------------------------------------------------

lad_lookup = (
    panel_df[["lad_uri", "lad_name"]]
    .drop_duplicates()
    .sort_values("lad_uri")
    .reset_index(drop=True)
)

lad_uris = lad_lookup["lad_uri"].tolist()
lad_to_idx = {lad: i for i, lad in enumerate(lad_uris)}
n_lads = len(lad_uris)


# ------------------------------------------------------------
# 2. Build scenario-aware temporal feature matrix
# ------------------------------------------------------------

def build_scenario_temporal_feature_matrix(panel_df, capacity_df, train_years, scenario):
    rows = []

    cap_s = (
        capacity_df[capacity_df["scenario"] == scenario]
        .set_index("lad_uri")
    )

    for lad_uri in lad_uris:

        part = (
            panel_df[panel_df["lad_uri"] == lad_uri]
            .sort_values("year")
        )

        values_by_year = {
            year: row
            for year, row in part.set_index("year").iterrows()
        }

        features = []

        for year in train_years:

            if year in values_by_year:
                keepership = values_by_year[year]["ev_keepership"]
                chargers = values_by_year[year]["ev_chargers"]
            else:
                keepership = 0
                chargers = 0

            features.extend([
                np.log1p(float(keepership)),
                np.log1p(float(chargers))
            ])

        if lad_uri in cap_s.index:
            K_keepership = cap_s.loc[lad_uri, "K_ev_keepership"]
            K_chargers = cap_s.loc[lad_uri, "K_ev_chargers"]
        else:
            K_keepership = 0
            K_chargers = 0

        features.extend([
            np.log1p(float(K_keepership)),
            np.log1p(float(K_chargers))
        ])

        rows.append(features)

    return np.asarray(rows, dtype=np.float32)


# ------------------------------------------------------------
# 3. Build similarity graph from scenario-aware LAD histories
# ------------------------------------------------------------

def build_similarity_adjacency(X, top_k=5):
    similarity = cosine_similarity(X)

    A = np.zeros_like(similarity, dtype=np.float32)

    for i in range(similarity.shape[0]):

        neighbours = np.argsort(similarity[i])[::-1]

        neighbours = [
            j for j in neighbours
            if j != i and similarity[i, j] > 0
        ][:top_k]

        for j in neighbours:
            A[i, j] = similarity[i, j]
            A[j, i] = similarity[i, j]

    np.fill_diagonal(A, 1.0)

    degree = np.sum(A, axis=1)
    degree[degree == 0] = 1.0

    D_inv_sqrt = np.diag(1.0 / np.sqrt(degree))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return A_norm.astype(np.float32)


# ------------------------------------------------------------
# 4. Build labels
# ------------------------------------------------------------

def build_labels(panel_df, target, test_year):
    y = np.full(n_lads, np.nan, dtype=np.float32)

    for lad_uri in lad_uris:

        part = panel_df[
            (panel_df["lad_uri"] == lad_uri) &
            (panel_df["year"] == test_year)
        ]

        if not part.empty:
            y[lad_to_idx[lad_uri]] = np.log1p(float(part[target].iloc[0]))

    return y


# ------------------------------------------------------------
# 5. Graph convolution layer
# ------------------------------------------------------------

class GraphConvolution(nn.Module):
    """Dense graph convolution: A_norm @ X @ W (+ activation). PyTorch port of the
    earlier Keras layer; PyTorch is what the setup cell installs, so no TensorFlow is needed."""

    def __init__(self, in_dim, units, activation=None):
        super().__init__()
        self.W = nn.Parameter(torch.empty(in_dim, units))
        nn.init.xavier_uniform_(self.W)          # glorot_uniform, as before
        self.activation = activation

    def forward(self, X, A):
        HW = A @ X @ self.W
        return self.activation(HW) if self.activation is not None else HW


class GCNForecaster(nn.Module):
    """Two-layer GCN (32, 16 units, ReLU) with a linear output, as in the original model."""

    def __init__(self, in_dim):
        super().__init__()
        self.gc1 = GraphConvolution(in_dim, 32, torch.relu)
        self.gc2 = GraphConvolution(32, 16, torch.relu)
        self.out = nn.Linear(16, 1)
        nn.init.xavier_uniform_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, X, A):
        return self.out(self.gc2(self.gc1(X, A), A))


def fit_scenario_aware_gcn(X, A_norm, y, epochs=1000, learning_rate=0.01):
    """Fit the GCN on log1p targets (masked MSE over the LADs with a label) and return
    rounded count predictions for every LAD."""
    valid_mask = ~np.isnan(y)

    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X).astype(np.float32)

    X_t = torch.tensor(X_scaled, dtype=torch.float32)
    A_t = torch.tensor(np.asarray(A_norm, dtype=np.float32))
    y_t = torch.tensor(np.nan_to_num(y, nan=0.0).reshape(-1, 1), dtype=torch.float32)
    mask_t = torch.tensor(valid_mask.reshape(-1, 1), dtype=torch.bool)

    torch.manual_seed(SEED)
    model = GCNForecaster(X.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_t, A_t)
        loss = torch.mean((y_t[mask_t] - pred[mask_t]) ** 2)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_log = model(X_t, A_t).numpy().ravel()

    # inverse log1p transform and round to counts
    pred = np.expm1(pred_log)
    pred = np.maximum(pred, 0)
    pred = np.rint(pred)

    return pred


# ------------------------------------------------------------
# 7. Walk-forward scenario-aware GNN backtesting
# ------------------------------------------------------------

prediction_store = {
    (scenario, lad_uri, target): {"y_true": [], "y_pred": []}
    for scenario in scenarios
    for lad_uri in lad_uris
    for target in targets
}

for scenario in scenarios:

    for test_year in test_years:

        train_years = sorted(
            panel_df.loc[
                panel_df["year"] < test_year,
                "year"
            ].unique()
        )

        if len(train_years) < 3:
            continue

        X = build_scenario_temporal_feature_matrix(
            panel_df=panel_df,
            capacity_df=capacity_df,
            train_years=train_years,
            scenario=scenario
        )

        A_norm = build_similarity_adjacency(
            X=X,
            top_k=top_k
        )

        for target in targets:

            y_log = build_labels(
                panel_df=panel_df,
                target=target,
                test_year=test_year
            )

            pred_all = fit_scenario_aware_gcn(
                X=X,
                A_norm=A_norm,
                y=y_log,
                epochs=epochs,
                learning_rate=learning_rate
            )

            for lad_uri in lad_uris:

                idx = lad_to_idx[lad_uri]

                true_row = panel_df[
                    (panel_df["lad_uri"] == lad_uri) &
                    (panel_df["year"] == test_year)
                ]

                if true_row.empty or np.isnan(y_log[idx]):
                    continue

                y_true = float(true_row[target].iloc[0])
                y_pred = float(np.asarray(pred_all[idx]).ravel()[0])

                prediction_store[(scenario, lad_uri, target)]["y_true"].append(y_true)
                prediction_store[(scenario, lad_uri, target)]["y_pred"].append(y_pred)


# ------------------------------------------------------------
# 8. LAD-level metrics
# ------------------------------------------------------------

metric_rows = []

for scenario in scenarios:

    for lad_uri in lad_uris:

        lad_name = lad_lookup.loc[
            lad_lookup["lad_uri"] == lad_uri,
            "lad_name"
        ].iloc[0]

        for target in targets:

            y_true_all = np.asarray(
                prediction_store[(scenario, lad_uri, target)]["y_true"],
                dtype=float
            )

            y_pred_all = np.asarray(
                prediction_store[(scenario, lad_uri, target)]["y_pred"],
                dtype=float
            )

            if len(y_true_all) == 0:
                continue

            mae = mean_absolute_error(y_true_all, y_pred_all)
            rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
            r2 = r2_score(y_true_all, y_pred_all) if len(y_true_all) > 1 else np.nan

            accuracy_percent = (
                max(0, r2 * 100)
                if not np.isnan(r2)
                else np.nan
            )

            metric_rows.append({
                "model": "scenario_aware_gnn",
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "target": target,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "accuracy_percent": accuracy_percent,
                "n_test_predictions": len(y_true_all)
            })


scenario_gnn_metrics_df = pd.DataFrame(metric_rows)

display(
    scenario_gnn_metrics_df.sort_values(
        ["target", "lad_name", "scenario"]
    )
)


# ------------------------------------------------------------
# 9. Summary table — same format as other models
# ------------------------------------------------------------

scenario_gnn_accuracy_summary_df = (
    scenario_gnn_metrics_df
    .groupby(["model", "scenario", "target"], as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean"),
        accuracy_percent=("accuracy_percent", "mean"),
        n_LADs=("lad_name", "nunique"),
        n_test_predictions=("n_test_predictions", "sum")
    )
)

scenario_gnn_accuracy_summary_df = (
    scenario_gnn_accuracy_summary_df
    .sort_values(["target", "mean_RMSE"])
    .reset_index()
)

display(scenario_gnn_accuracy_summary_df)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

scenario_gnn_metrics_df.to_csv(
    "scenario_aware_gnn_lad_level_accuracy_metrics.csv",
    index=False
)

scenario_gnn_accuracy_summary_df.to_csv(
    "scenario_aware_gnn_summary_accuracy_metrics.csv",
    index=False
)

print("Saved:")
print("- scenario_aware_gnn_lad_level_accuracy_metrics.csv")
print("- scenario_aware_gnn_summary_accuracy_metrics.csv")

Generate EV charger demand predictions through 2045 using the best-performing model.

In [ ]:
# ------------------------------------------------------------
# Generate EV charger predictions to 2045 only
# ------------------------------------------------------------

forecast_years = list(range(2026, 2046))
target = "ev_chargers"

forecast_rows = []

for scenario in scenarios:

    for forecast_year in forecast_years:

        train_years = sorted(
            panel_df.loc[
                panel_df["year"] < forecast_year,
                "year"
            ].unique()
        )

        if len(train_years) < 3:
            continue

        X = build_scenario_temporal_feature_matrix(
            panel_df=panel_df,
            capacity_df=capacity_df,
            train_years=train_years,
            scenario=scenario
        )

        A_norm = build_similarity_adjacency(
            X=X,
            top_k=top_k
        )

        # Use latest available EV charger values as pseudo-labels for training
        latest_year = max(train_years)

        y_log = build_labels(
            panel_df=panel_df,
            target=target,
            test_year=latest_year
        )

        pred_all = fit_scenario_aware_gcn(
            X=X,
            A_norm=A_norm,
            y=y_log,
            epochs=epochs,
            learning_rate=learning_rate
        )

        for lad_uri in lad_uris:

            idx = lad_to_idx[lad_uri]

            lad_name = lad_lookup.loc[
                lad_lookup["lad_uri"] == lad_uri,
                "lad_name"
            ].iloc[0]

            forecast_rows.append({
                "model": "scenario_aware_gnn",
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "year": forecast_year,
                "target": "ev_chargers",
                "predicted_ev_chargers": float(pred_all[idx])
            })

ev_charger_predictions_2045_df = pd.DataFrame(forecast_rows)

display(
    ev_charger_predictions_2045_df.sort_values(
        ["scenario", "lad_name", "year"]
    )
)

ev_charger_predictions_2045_df.to_csv(
    "ev_charger_predictions_to_2045.csv",
    index=False
)

print("Saved: ev_charger_predictions_to_2045.csv")

In [ ]:
files.download("ev_charger_predictions_to_2045.csv")

In [ ]:
# ============================================================
# Wales LAD map:
# Keepers per charger + deprivation decile
# Publication-quality version
# ============================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
import requests

from matplotlib.colors import ListedColormap, BoundaryNorm
from google.colab import files

# ------------------------------------------------------------
# 1. Download ONS LAD boundaries
# ------------------------------------------------------------

geojson_url = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Local_Authority_Districts_December_2024_Boundaries_UK_BUC/"
    "FeatureServer/0/query?"
    "where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

r = requests.get(geojson_url, timeout=120)
r.raise_for_status()

with open("lad_2024_uk_buc.geojson", "wb") as f:
    f.write(r.content)

uk_lad_gdf = gpd.read_file("lad_2024_uk_buc.geojson")

# ------------------------------------------------------------
# 2. Detect and standardise columns
# ------------------------------------------------------------

code_col = [c for c in uk_lad_gdf.columns if "CD" in c.upper() or "CODE" in c.upper()][0]
name_col = [c for c in uk_lad_gdf.columns if "NM" in c.upper() or "NAME" in c.upper()][0]

wales_lad_gdf = uk_lad_gdf[
    uk_lad_gdf[code_col].astype(str).str.startswith("W")
].copy()

wales_lad_gdf = wales_lad_gdf.rename(columns={name_col: "lad_name"})

# ------------------------------------------------------------
# 3. Data
# ------------------------------------------------------------

data_df = pd.DataFrame({
    "lad_name": [
        "Neath Port Talbot", "Flintshire", "Vale of Glamorgan",
        "Caerphilly", "Bridgend", "Cardiff", "Torfaen",
        "Monmouthshire", "Swansea", "Wrexham", "Denbighshire",
        "Conwy", "Rhondda Cynon Taf", "Carmarthenshire",
        "Merthyr Tydfil", "Powys", "Newport",
        "Isle of Anglesey", "Pembrokeshire",
        "Blaenau Gwent", "Ceredigion", "Gwynedd"
    ],
    "ratio": [
        18.1, 14.6, 14.1, 12.4, 11.2, 10.4, 10.4,
        9.5, 9.2, 9.1, 8.9, 8.6, 7.8, 7.2,
        6.4, 5.7, 5.6, 4.9, 4.3, 3.9, 3.6, 2.4
    ],
    "deprivation_decile": [
        2, 5, 9, 3, 4, 6, 3,
        10, 4, 5, 4, 6, 2, 5,
        1, 8, 3, 6, 5, 1, 7, 7
    ]
})

# ------------------------------------------------------------
# 4. Short names for labels
# ------------------------------------------------------------

short_names = {
    "Neath Port Talbot": "NPT",
    "Vale of Glamorgan": "Vale Glam.",
    "Rhondda Cynon Taf": "RCT",
    "Isle of Anglesey": "Anglesey",
    "Blaenau Gwent": "B. Gwent",
    "Merthyr Tydfil": "Merthyr",
    "Carmarthenshire": "Carmarth.",
    "Pembrokeshire": "Pembrok.",
    "Monmouthshire": "Monmouth.",
    "Denbighshire": "Denbigh.",
    "Caerphilly": "Caerph.",
    "Ceredigion": "Ceredig."
}

data_df["label_name"] = data_df["lad_name"].replace(short_names)

# ------------------------------------------------------------
# 5. Merge geography
# ------------------------------------------------------------

map_df = wales_lad_gdf.merge(
    data_df,
    on="lad_name",
    how="left"
)

print("Matched LADs:", map_df["ratio"].notna().sum(), "of", len(data_df))

# ------------------------------------------------------------
# 6. Pressure categories
# ------------------------------------------------------------

def pressure_category(x):
    if x >= 18:
        return 4
    elif x >= 14:
        return 3
    elif x >= 10:
        return 2
    elif x >= 6:
        return 1
    else:
        return 0

map_df["pressure_class"] = map_df["ratio"].apply(pressure_category)

pastel_colors = [
    "#d9f0c1",
    "#fff2b2",
    "#ffd59e",
    "#fdae8b",
    "#f4a3a8"
]

cmap = ListedColormap(pastel_colors)
norm = BoundaryNorm([0, 1, 2, 3, 4, 5], cmap.N)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 13))

map_df.plot(
    column="pressure_class",
    cmap=cmap,
    norm=norm,
    linewidth=1.6,
    edgecolor="black",
    ax=ax,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)

# ------------------------------------------------------------
# 8. Label offsets
# ------------------------------------------------------------

offsets = {
    "Cardiff": (0.04, -0.03),
    "Newport": (0.04, 0.02),
    "Blaenau Gwent": (0.02, 0.03),
    "Merthyr Tydfil": (-0.02, 0.04),
    "Torfaen": (0.03, 0.04),
    "Caerphilly": (-0.03, -0.02),
    "Vale of Glamorgan": (-0.04, -0.03),
    "Bridgend": (-0.04, -0.02),
    "Rhondda Cynon Taf": (-0.03, 0.03),
    "Neath Port Talbot": (-0.04, 0.00),
    "Swansea": (-0.04, -0.02),
}

# ------------------------------------------------------------
# 9. Labels
# ------------------------------------------------------------

for _, row in map_df.iterrows():

    if row.geometry is None or pd.isna(row["ratio"]):
        continue

    pt = row.geometry.representative_point()

    dx, dy = offsets.get(row["lad_name"], (0, 0))

    label = (
        f"{row['label_name']}\n"
        f"{row['ratio']:.1f}:1\n"
        f"D{int(row['deprivation_decile'])}"
    )

    txt = ax.text(
        pt.x + dx,
        pt.y + dy,
        label,
        fontsize=6.5,
        ha="center",
        va="center",
        fontweight="bold",
        color="black",
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="none",
            edgecolor="none"
        ),
        zorder=10
    )

    txt.set_path_effects([
        pe.withStroke(linewidth=3, foreground="white")
    ])

# ------------------------------------------------------------
# 10. Legends
# ------------------------------------------------------------

ratio_legend = [
    mpatches.Patch(facecolor="#f4a3a8", edgecolor="black", label="18 or more"),
    mpatches.Patch(facecolor="#fdae8b", edgecolor="black", label="14–17.9"),
    mpatches.Patch(facecolor="#ffd59e", edgecolor="black", label="10–13.9"),
    mpatches.Patch(facecolor="#fff2b2", edgecolor="black", label="6–9.9"),
    mpatches.Patch(facecolor="#d9f0c1", edgecolor="black", label="2–5.9"),
]

legend1 = ax.legend(
    handles=ratio_legend,
    title="Keepers per charger",
    loc="upper left",
    bbox_to_anchor=(-0.03, 0.98),
    frameon=True,
    fontsize=9,
    title_fontsize=10
)

ax.add_artist(legend1)

pressure_legend = [
    mpatches.Patch(
        facecolor="#f4a3a8",
        edgecolor="black",
        label="High infrastructure pressure"
    ),
    mpatches.Patch(
        facecolor="#ffd59e",
        edgecolor="black",
        label="Moderate infrastructure pressure"
    )
]

legend2 = ax.legend(
    handles=pressure_legend,
    loc="upper left",
    bbox_to_anchor=(-0.03, 0.70),
    frameon=True,
    fontsize=8,
    title="Interpretation",
    title_fontsize=9
)

# ------------------------------------------------------------
# 11. Title and note
# ------------------------------------------------------------

ax.set_title(
    "Electric Vehicle (EV) Keepers per Charger Across Welsh LADs\n"
    "(January 2026 Public Chargers / December 2025 Private EV Keeperships)",
    fontsize=15,
    fontweight="bold",
    pad=22
)

note = (
    "Labels show LAD name, keepers-per-charger ratio, and deprivation decile.\n"
    "Deprivation decile: 1 = most deprived; 10 = least deprived."
)

ax.text(
    0.02,
    0.02,
    note,
    transform=ax.transAxes,
    fontsize=6,
    va="bottom",
    ha="left",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="black",
        alpha=0.85
    )
)

# ------------------------------------------------------------
# 12. Prevent clipping
# ------------------------------------------------------------

xmin, ymin, xmax, ymax = map_df.total_bounds

x_pad = (xmax - xmin) * 0.12
y_pad = (ymax - ymin) * 0.08

ax.set_xlim(xmin - x_pad, xmax + x_pad)
ax.set_ylim(ymin - y_pad, ymax + y_pad)

ax.set_axis_off()

# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

plt.subplots_adjust(
    left=0.08,
    right=0.95,
    top=0.92,
    bottom=0.06
)

plt.tight_layout(pad=2)

plt.savefig(
    "wales_keepers_per_charger_deprivation_map_publication.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "wales_keepers_per_charger_deprivation_map_publication.pdf",
    bbox_inches="tight"
)

plt.show()

files.download("wales_keepers_per_charger_deprivation_map_publication.png")
files.download("wales_keepers_per_charger_deprivation_map_publication.pdf")



In [ ]:
# ============================================================
# Wales LAD map:
# Predicted 2045 keepers per charger + deprivation decile
# ============================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
import requests

from matplotlib.colors import ListedColormap, BoundaryNorm
from google.colab import files

# ------------------------------------------------------------
# 1. Download ONS LAD boundaries
# ------------------------------------------------------------

geojson_url = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Local_Authority_Districts_December_2024_Boundaries_UK_BUC/"
    "FeatureServer/0/query?"
    "where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

r = requests.get(geojson_url, timeout=120)
r.raise_for_status()

with open("lad_2024_uk_buc.geojson", "wb") as f:
    f.write(r.content)

uk_lad_gdf = gpd.read_file("lad_2024_uk_buc.geojson")

# ------------------------------------------------------------
# 2. Detect and standardise columns
# ------------------------------------------------------------

code_col = [c for c in uk_lad_gdf.columns if "CD" in c.upper() or "CODE" in c.upper()][0]
name_col = [c for c in uk_lad_gdf.columns if "NM" in c.upper() or "NAME" in c.upper()][0]

wales_lad_gdf = uk_lad_gdf[
    uk_lad_gdf[code_col].astype(str).str.startswith("W")
].copy()

wales_lad_gdf = wales_lad_gdf.rename(columns={name_col: "lad_name"})

# ------------------------------------------------------------
# 3. Predicted 2045 low-scenario ratio data
# ------------------------------------------------------------

data_df = pd.DataFrame({
    "lad_name": [
        "Cardiff", "Neath Port Talbot", "Swansea", "Rhondda Cynon Taf",
        "Bridgend", "Flintshire", "Caerphilly", "Carmarthenshire",
        "Vale of Glamorgan", "Newport", "Wrexham", "Monmouthshire",
        "Torfaen", "Conwy", "Denbighshire", "Powys", "Merthyr Tydfil",
        "Pembrokeshire", "Blaenau Gwent", "Gwynedd",
        "Isle of Anglesey", "Ceredigion"
    ],
    "ratio": [
        687.3, 493.4, 404.1, 323.0,
        293.1, 235.2, 210.5, 201.9,
        144.2, 119.3, 101.4, 88.7,
        84.3, 82.9, 78.7, 61.7, 46.2,
        44.6, 41.3, 41.1,
        23.2, 18.2
    ],
    "deprivation_decile": [
        6, 2, 4, 2,
        4, 5, 3, 5,
        9, 3, 5, 10,
        3, 6, 4, 8, 1,
        5, 1, 7,
        6, 7
    ]
})

# ------------------------------------------------------------
# 4. Short names for labels
# ------------------------------------------------------------

short_names = {
    "Neath Port Talbot": "NPT",
    "Vale of Glamorgan": "Vale Glam.",
    "Rhondda Cynon Taf": "RCT",
    "Isle of Anglesey": "Anglesey",
    "Blaenau Gwent": "B. Gwent",
    "Merthyr Tydfil": "Merthyr",
    "Carmarthenshire": "Carmarth.",
    "Pembrokeshire": "Pembrok.",
    "Monmouthshire": "Monmouth.",
    "Denbighshire": "Denbigh.",
    "Caerphilly": "Caerph.",
    "Ceredigion": "Ceredig."
}

data_df["label_name"] = data_df["lad_name"].replace(short_names)

# ------------------------------------------------------------
# 5. Merge geography
# ------------------------------------------------------------

map_df = wales_lad_gdf.merge(
    data_df,
    on="lad_name",
    how="left"
)

print("Matched LADs:", map_df["ratio"].notna().sum(), "of", len(data_df))

# ------------------------------------------------------------
# 6. Predicted pressure categories
# ------------------------------------------------------------

def pressure_category(x):
    if x >= 500:
        return 4
    elif x >= 250:
        return 3
    elif x >= 100:
        return 2
    elif x >= 50:
        return 1
    else:
        return 0

map_df["pressure_class"] = map_df["ratio"].apply(pressure_category)

pastel_colors = [
    "#fff2b2",  # 18–49.9
    "#ffd59e",  # 50–99.9
    "#fdae8b",  # 100–249.9
    "#f4a3a8",  # 250–499.9
    "#c75b7a"   # 500+
]

cmap = ListedColormap(pastel_colors)
norm = BoundaryNorm([0, 1, 2, 3, 4, 5], cmap.N)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 13))

map_df.plot(
    column="pressure_class",
    cmap=cmap,
    norm=norm,
    linewidth=1.6,
    edgecolor="black",
    ax=ax,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)

# ------------------------------------------------------------
# 8. Label offsets
# ------------------------------------------------------------

offsets = {
    "Cardiff": (0.04, -0.03),
    "Newport": (0.04, 0.02),
    "Blaenau Gwent": (0.02, 0.03),
    "Merthyr Tydfil": (-0.02, 0.04),
    "Torfaen": (0.03, 0.04),
    "Caerphilly": (-0.03, -0.02),
    "Vale of Glamorgan": (-0.04, -0.03),
    "Bridgend": (-0.04, -0.02),
    "Rhondda Cynon Taf": (-0.03, 0.03),
    "Neath Port Talbot": (-0.04, 0.00),
    "Swansea": (-0.04, -0.02),
}

# ------------------------------------------------------------
# 9. Labels
# ------------------------------------------------------------

for _, row in map_df.iterrows():

    if row.geometry is None or pd.isna(row["ratio"]):
        continue

    pt = row.geometry.representative_point()
    dx, dy = offsets.get(row["lad_name"], (0, 0))

    label = (
        f"{row['label_name']}\n"
        f"{row['ratio']:.1f}:1\n"
        f"D{int(row['deprivation_decile'])}"
    )

    txt = ax.text(
        pt.x + dx,
        pt.y + dy,
        label,
        fontsize=6.5,
        ha="center",
        va="center",
        fontweight="bold",
        color="black",
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="none",
            edgecolor="none"
        ),
        zorder=10
    )

    txt.set_path_effects([
        pe.withStroke(linewidth=3, foreground="white")
    ])

# ------------------------------------------------------------
# 10. Legend inside figure
# ------------------------------------------------------------

ratio_legend = [
    mpatches.Patch(facecolor="#c75b7a", edgecolor="black", label="500 or more"),
    mpatches.Patch(facecolor="#f4a3a8", edgecolor="black", label="250–499.9"),
    mpatches.Patch(facecolor="#fdae8b", edgecolor="black", label="100–249.9"),
    mpatches.Patch(facecolor="#ffd59e", edgecolor="black", label="50–99.9"),
    mpatches.Patch(facecolor="#fff2b2", edgecolor="black", label="18–49.9"),
]

legend1 = ax.legend(
    handles=ratio_legend,
    title="Keepers per charger (2045)",
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    frameon=False,
    fontsize=8,
    title_fontsize=9
)

ax.add_artist(legend1)

# ------------------------------------------------------------
# 11. Title
# ------------------------------------------------------------

ax.set_title(
    "Predicted Electric Vehicle (EV) Keepers per Charger Across Welsh LADs, 2045\n"
    "(Low-growth scenario)",
    fontsize=15,
    fontweight="bold",
    pad=22
)

# ------------------------------------------------------------
# 12. Prevent clipping
# ------------------------------------------------------------

xmin, ymin, xmax, ymax = map_df.total_bounds

x_pad = (xmax - xmin) * 0.12
y_pad = (ymax - ymin) * 0.08

ax.set_xlim(xmin - x_pad, xmax + x_pad)
ax.set_ylim(ymin - y_pad, ymax + y_pad)

ax.set_axis_off()

# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.92,
    bottom=0.02
)

plt.tight_layout(pad=2)

plt.savefig(
    "wales_predicted_2045_keepers_per_charger_map.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "wales_predicted_2045_keepers_per_charger_map.pdf",
    bbox_inches="tight"
)

plt.show()

files.download("wales_predicted_2045_keepers_per_charger_map.png")
files.download("wales_predicted_2045_keepers_per_charger_map.pdf")